# CKM3N — Centro 4128 — Datalake

**Tabela:** `dev_procurement.corp_curated.vw_ds_ctrl_ckm3n`
**Transação SAP:** CKM3N · **Objeto:** **VIEW**
**Domínio:** Ledger de material (material ledger)
**Filtro do cenário:** `cod_area_avaliacao = '4128'`
**Colunas:** 99 · **Clustering declarado:** _(nenhum)_

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** — cada uma consulta a tabela
diretamente com o filtro do centro embutido. Não há widget, view temporária nem ordem obrigatória.

## ⚠️ Coluna de recorte — leia antes de rodar

Esta tabela **não possui `cod_centro`**. O filtro usa `cod_area_avaliacao` (valuation area), que no SAP normalmente **coincide com o centro** quando o nível de avaliação é por planta. Confirme na seção 0 antes de prosseguir.

**Filtro aplicado:** ``cod_area_avaliacao` = '4128'`

## Objetivo

Extrair e caracterizar **toda** a base do centro 4128 para comparação com o extrato do SAP.

## Seções

| # | Conteúdo |
|---|---|
| **0** | **Descoberta da coluna de recorte** |
| 1 | Metadados |
| 2 | Volumetria e representatividade |
| 3 | Confirmação do filtro |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| **6** | **Preenchimento de todas as colunas** |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| 9 | Flags booleanas |
| 10 | Perfil numérico |
| **11** | **Totais para conciliação com o SAP** |
| 12 | Datas e timestamps |
| 13 | Códigos e zeros à esquerda |
| **14** | **Chaves normalizadas para join** |
| **15** | **Checksum de linha** |
| 16 | Amostra |
| 17 | Distribuição interna |
| **18** | **Análises específicas da transação** |
| **19** | **EXTRAÇÃO COMPLETA** |
| 20 | Resumo do cenário |

> **Aviso:** contagem de linhas não é evidência de qualidade. Ver seções 4, 5 e 15.


## 0. Descoberta da coluna de recorte

**Execute esta seção primeiro.** Ela mostra quais valores de `cod_area_avaliacao` existem e se algum
se relaciona com o centro 4128.

Se a primeira consulta voltar vazia, o filtro deste notebook **não vai retornar dados** —
ajuste o critério antes de seguir.

In [0]:
-- 0. VALORES DE cod_area_avaliacao RELACIONADOS AO CENTRO 4128
SELECT CAST(`cod_area_avaliacao` AS STRING)          AS valor,
       length(CAST(`cod_area_avaliacao` AS STRING))  AS comprimento,
       COUNT(*)                         AS linhas
FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n
WHERE trim(CAST(`cod_area_avaliacao` AS STRING)) = '4128'
   OR trim(CAST(`cod_area_avaliacao` AS STRING)) LIKE '4128%'
   OR trim(CAST(`cod_area_avaliacao` AS STRING)) LIKE '%4128%'
GROUP BY CAST(`cod_area_avaliacao` AS STRING), length(CAST(`cod_area_avaliacao` AS STRING))
ORDER BY linhas DESC
LIMIT 50;

In [0]:
-- 0.1 TODOS OS VALORES DE cod_area_avaliacao (top 60 por volume)
SELECT CAST(`cod_area_avaliacao` AS STRING) AS valor,
       COUNT(*)                AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n
GROUP BY CAST(`cod_area_avaliacao` AS STRING)
ORDER BY linhas DESC
LIMIT 60;

## 1. Metadados

Este objeto é uma **view**. `DESCRIBE DETAIL` e `DESCRIBE HISTORY` só funcionam em tabelas
Delta — por isso ficaram comentados, para não interromper o **Run All**.

In [0]:
-- 1. ESTRUTURA
DESCRIBE EXTENDED dev_procurement.corp_curated.vw_ds_ctrl_ckm3n;

In [0]:
-- 1.1 Detalhe e historico nao se aplicam a view.
-- Descomente apenas se o objeto for convertido em tabela Delta.
-- DESCRIBE DETAIL dev_procurement.corp_curated.vw_ds_ctrl_ckm3n;
-- DESCRIBE HISTORY dev_procurement.corp_curated.vw_ds_ctrl_ckm3n LIMIT 20;
SELECT 'Objeto e uma VIEW — os dois comandos acima nao se aplicam' AS observacao;

## 2. Volumetria e representatividade

Quanto o centro 4128 representa do total.

In [0]:
-- 2. VOLUMETRIA DO CENARIO
SELECT
  (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n)                            AS linhas_tabela_toda,
  (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128')                        AS linhas_do_cenario,
  ROUND(100.0 * (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128')
              / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n), 4)           AS pct_do_total,
  (SELECT COUNT(DISTINCT `cod_area_avaliacao`) FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n)              AS valores_distintos_na_tabela;

## 3. Confirmação do filtro

Confirma que o valor `4128` existe e que não há variação de formato (espaços,
zeros à esquerda) que faça o filtro perder linhas silenciosamente.

**Se retornar mais de uma linha com formatos diferentes, o filtro está incompleto.**

In [0]:
-- 3. O FILTRO PEGOU TUDO?
SELECT CAST(`cod_area_avaliacao` AS STRING)         AS valor_bruto,
       length(CAST(`cod_area_avaliacao` AS STRING)) AS comprimento,
       COUNT(*)                        AS linhas
FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n
WHERE regexp_replace(trim(CAST(`cod_area_avaliacao` AS STRING)), '^0+', '') = '4128'
   OR trim(CAST(`cod_area_avaliacao` AS STRING)) = '4128'
GROUP BY CAST(`cod_area_avaliacao` AS STRING), length(CAST(`cod_area_avaliacao` AS STRING))
ORDER BY linhas DESC;

## 4. Granularidade e chave real

`linhas ÷ chaves distintas`. Razão maior que 1,00 indica dimensão adicional
multiplicando as linhas.

In [0]:
-- GRANULARIDADE NO CENARIO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'),
g AS (
  SELECT 'num_ledger_material' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_ledger_material` FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128')
UNION ALL
  SELECT 'num_ledger_material + num_item_ledger_material' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_ledger_material`, `num_item_ledger_material` FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128')
UNION ALL
  SELECT 'num_ledger_material + num_item_ledger_material + num_ledger_material_modificacao_preco_ano' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_ledger_material`, `num_item_ledger_material`, `num_ledger_material_modificacao_preco_ano` FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128')
UNION ALL
  SELECT 'cod_material + cod_area_avaliacao + cod_ano_periodo_aaaappp_lancamento' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_material`, `cod_area_avaliacao`, `cod_ano_periodo_aaaappp_lancamento` FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128')
)
SELECT g.chave, t.total AS linhas, g.distintos,
       ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade

Analisando pela chave `num_ledger_material + num_item_ledger_material`.

**Regra:** linhas idênticas = duplicata real (erro de carga).
Linhas distintas = granularidade adicional legítima.

In [0]:
-- 5. CHAVES DUPLICADAS
SELECT `num_ledger_material`, `num_item_ledger_material`, COUNT(*) AS qtd
FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
GROUP BY `num_ledger_material`, `num_item_ledger_material`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 30;

In [0]:
-- O QUE DIFERENCIA AS LINHAS DUPLICADAS
WITH cen AS (
  SELECT * FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
),
dup AS (
  SELECT `num_ledger_material`, `num_item_ledger_material` FROM cen GROUP BY `num_ledger_material`, `num_item_ledger_material` HAVING COUNT(*) > 1
),
d AS (
  SELECT c.* FROM cen c JOIN dup ON c.`num_ledger_material` <=> dup.`num_ledger_material` AND c.`num_item_ledger_material` <=> dup.`num_item_ledger_material`
),
agg AS (
  SELECT `num_ledger_material`, `num_item_ledger_material`,
         COUNT(DISTINCT `cod_material`) AS `cod_material`,
         COUNT(DISTINCT `cod_area_avaliacao`) AS `cod_area_avaliacao`,
         COUNT(DISTINCT `cod_referencia_documento`) AS `cod_referencia_documento`,
         COUNT(DISTINCT `tp_moeda_externa_perspectiva_avaliacao`) AS `tp_moeda_externa_perspectiva_avaliacao`,
         COUNT(DISTINCT `cod_ledger_contabilidade_geral`) AS `cod_ledger_contabilidade_geral`,
         COUNT(DISTINCT `cod_referencia_execucao_calculo_custos`) AS `cod_referencia_execucao_calculo_custos`,
         COUNT(DISTINCT `cod_principio_contabil`) AS `cod_principio_contabil`,
         COUNT(DISTINCT `num_calculo_custos_sem_estrutura_qtd`) AS `num_calculo_custos_sem_estrutura_qtd`,
         COUNT(DISTINCT `cod_ano_periodo_aaaappp_lancamento`) AS `cod_ano_periodo_aaaappp_lancamento`,
         COUNT(DISTINCT `cod_ano_periodo_origem_material_processo`) AS `cod_ano_periodo_origem_material_processo`,
         COUNT(DISTINCT `cod_ano_corrente_periodo_yyyyppp_momento`) AS `cod_ano_corrente_periodo_yyyyppp_momento`,
         COUNT(DISTINCT `ind_calculo_custos`) AS `ind_calculo_custos`,
         COUNT(DISTINCT `cod_operacao_empresarial`) AS `cod_operacao_empresarial`,
         COUNT(DISTINCT `tp_ledger_material_operacao`) AS `tp_ledger_material_operacao`,
         COUNT(DISTINCT `tp_item_documento_ledger_material`) AS `tp_item_documento_ledger_material`,
         COUNT(DISTINCT `tp_exibicao_dados_ledger_material`) AS `tp_exibicao_dados_ledger_material`,
         COUNT(DISTINCT `tp_processo_alternativa_suprimento_consumo`) AS `tp_processo_alternativa_suprimento_consumo`,
         COUNT(DISTINCT `cod_grupo_tipos_movimento_ledger_material`) AS `cod_grupo_tipos_movimento_ledger_material`,
         COUNT(DISTINCT `cod_alternativa_processo_suprimento`) AS `cod_alternativa_processo_suprimento`,
         COUNT(DISTINCT `cod_processo_producao`) AS `cod_processo_producao`,
         COUNT(DISTINCT `num_calculo_custos_quantidade_produzida`) AS `num_calculo_custos_quantidade_produzida`,
         COUNT(DISTINCT `qt_transacao_unidade_medida_avaliacao`) AS `qt_transacao_unidade_medida_avaliacao`,
         COUNT(DISTINCT `vl_estoque_operacional`) AS `vl_estoque_operacional`,
         COUNT(DISTINCT `vl_diferenca_preco`) AS `vl_diferenca_preco`,
         COUNT(DISTINCT `vl_desvio_taxa_cambio`) AS `vl_desvio_taxa_cambio`,
         COUNT(DISTINCT `vl_diferenca_transferencia`) AS `vl_diferenca_transferencia`,
         COUNT(DISTINCT `vl_estoque_alternativo_interno_periodico`) AS `vl_estoque_alternativo_interno_periodico`,
         COUNT(DISTINCT `qt_estoque_avaliado`) AS `qt_estoque_avaliado`,
         COUNT(DISTINCT `vl_estoque_avaliado_total_antes_transacao`) AS `vl_estoque_avaliado_total_antes_transacao`,
         COUNT(DISTINCT `vl_alternativo_estoque_avaliado_total_antes`) AS `vl_alternativo_estoque_avaliado_total_antes`,
         COUNT(DISTINCT `vl_preco_padrao_antes_transacao`) AS `vl_preco_padrao_antes_transacao`,
         COUNT(DISTINCT `qt_unidade_preco_antes_transacao`) AS `qt_unidade_preco_antes_transacao`,
         COUNT(DISTINCT `vl_venda_calculo_real_artigo`) AS `vl_venda_calculo_real_artigo`,
         COUNT(DISTINCT `qt_limitacao_preco`) AS `qt_limitacao_preco`,
         COUNT(DISTINCT `vl_lucro_intercompanhias`) AS `vl_lucro_intercompanhias`,
         COUNT(DISTINCT `vl_estoque_venda`) AS `vl_estoque_venda`,
         COUNT(DISTINCT `vl_coeficiente_equivalencia_producao_conjunta`) AS `vl_coeficiente_equivalencia_producao_conjunta`,
         COUNT(DISTINCT `vl_preco_medio_movel_antes_transacao`) AS `vl_preco_medio_movel_antes_transacao`,
         COUNT(DISTINCT `dh_registro_hora`) AS `dh_registro_hora`,
         COUNT(DISTINCT `qt_pedido_unidade_medida_pedido`) AS `qt_pedido_unidade_medida_pedido`,
         COUNT(DISTINCT `vl_entrada_fatura_ordem_producao`) AS `vl_entrada_fatura_ordem_producao`,
         COUNT(DISTINCT `vl_liquidacao_ordem_producao`) AS `vl_liquidacao_ordem_producao`,
         COUNT(DISTINCT `vl_estoque_operacional_centro_lucro`) AS `vl_estoque_operacional_centro_lucro`,
         COUNT(DISTINCT `qt_referencia_fatura_liq_ordem_producao`) AS `qt_referencia_fatura_liq_ordem_producao`,
         COUNT(DISTINCT `vl_diferenca_preco_centro_lucro`) AS `vl_diferenca_preco_centro_lucro`,
         COUNT(DISTINCT `sg_quantidade_avaliacao`) AS `sg_quantidade_avaliacao`,
         COUNT(DISTINCT `cod_moeda`) AS `cod_moeda`,
         COUNT(DISTINCT `ind_preco_desvio_taxa_cambio_relevantes`) AS `ind_preco_desvio_taxa_cambio_relevantes`,
         COUNT(DISTINCT `ind_desvios_sao_relevantes_logica_limitacao`) AS `ind_desvios_sao_relevantes_logica_limitacao`,
         COUNT(DISTINCT `ind_preco_interno_relevante_modificacao_centro`) AS `ind_preco_interno_relevante_modificacao_centro`,
         COUNT(DISTINCT `dt_lancamento_documento`) AS `dt_lancamento_documento`,
         COUNT(DISTINCT `num_ciclo_execucao`) AS `num_ciclo_execucao`,
         COUNT(DISTINCT `num_ordem`) AS `num_ordem`,
         COUNT(DISTINCT `num_item_ordem`) AS `num_item_ordem`,
         COUNT(DISTINCT `num_ledger_material_modificacao_preco_ano`) AS `num_ledger_material_modificacao_preco_ano`,
         COUNT(DISTINCT `cod_operacao_referencia`) AS `cod_operacao_referencia`,
         COUNT(DISTINCT `cod_sistema_logico_origem`) AS `cod_sistema_logico_origem`,
         COUNT(DISTINCT `sg_unidades_organizacionais_referencia`) AS `sg_unidades_organizacionais_referencia`,
         COUNT(DISTINCT `num_documento_referencia`) AS `num_documento_referencia`,
         COUNT(DISTINCT `num_partida_individual_referencia`) AS `num_partida_individual_referencia`,
         COUNT(DISTINCT `num_grupo_partidas_individuais_referencia`) AS `num_grupo_partidas_individuais_referencia`,
         COUNT(DISTINCT `nm_usuario`) AS `nm_usuario`,
         COUNT(DISTINCT `cod_transacao`) AS `cod_transacao`,
         COUNT(DISTINCT `cod_operacao_ledger_material`) AS `cod_operacao_ledger_material`,
         COUNT(DISTINCT `tp_transacao_pedido_historico_ordem_producao`) AS `tp_transacao_pedido_historico_ordem_producao`,
         COUNT(DISTINCT `cod_grupo_origem_classe_custo`) AS `cod_grupo_origem_classe_custo`,
         COUNT(DISTINCT `cod_classe_custo_ordem`) AS `cod_classe_custo_ordem`,
         COUNT(DISTINCT `tp_movimento_administracao_estoques`) AS `tp_movimento_administracao_estoques`,
         COUNT(DISTINCT `cod_centro_ordem_producao_dados_historico`) AS `cod_centro_ordem_producao_dados_historico`,
         COUNT(DISTINCT `ind_estorno_apropriacao_ordem`) AS `ind_estorno_apropriacao_ordem`,
         COUNT(DISTINCT `sg_unidade_medida_ordem_producao`) AS `sg_unidade_medida_ordem_producao`,
         COUNT(DISTINCT `cod_periodo_apropriacao_custos_ordem`) AS `cod_periodo_apropriacao_custos_ordem`,
         COUNT(DISTINCT `ind_capitalizar_diferencas_cambio`) AS `ind_capitalizar_diferencas_cambio`,
         COUNT(DISTINCT `dt_envelhecimento_dados`) AS `dt_envelhecimento_dados`,
         COUNT(DISTINCT `cod_motivo_classificacao_contabil`) AS `cod_motivo_classificacao_contabil`,
         COUNT(DISTINCT `cod_motivo_modificacao`) AS `cod_motivo_modificacao`,
         COUNT(DISTINCT `cod_versao_migracao_dados`) AS `cod_versao_migracao_dados`,
         COUNT(DISTINCT `st_cobertura_estoque`) AS `st_cobertura_estoque`,
         COUNT(DISTINCT `cod_caso_fallback`) AS `cod_caso_fallback`,
         COUNT(DISTINCT `tp_preco_fallback`) AS `tp_preco_fallback`,
         COUNT(DISTINCT `cod_avaliacao_externa`) AS `cod_avaliacao_externa`,
         COUNT(DISTINCT `ind_conexao_interrompida`) AS `ind_conexao_interrompida`,
         COUNT(DISTINCT `cod_moeda_centro_lucro`) AS `cod_moeda_centro_lucro`,
         COUNT(DISTINCT `num_documento_compras`) AS `num_documento_compras`,
         COUNT(DISTINCT `num_item_documento_compras`) AS `num_item_documento_compras`,
         COUNT(DISTINCT `num_exercicio_periodo_atual`) AS `num_exercicio_periodo_atual`,
         COUNT(DISTINCT `num_documento_referencia_historico`) AS `num_documento_referencia_historico`,
         COUNT(DISTINCT `num_item_documento_referencia_historico`) AS `num_item_documento_referencia_historico`,
         COUNT(DISTINCT `tp_condicao`) AS `tp_condicao`,
         COUNT(DISTINCT `desc_categoria_ledger_material`) AS `desc_categoria_ledger_material`,
         COUNT(DISTINCT `dt_sistema`) AS `dt_sistema`,
         COUNT(DISTINCT `dh_carga`) AS `dh_carga`,
         COUNT(DISTINCT `dh_atualizacao`) AS `dh_atualizacao`,
         COUNT(DISTINCT `hash_conteudo`) AS `hash_conteudo`,
         COUNT(DISTINCT `vl_estoque_real`) AS `vl_estoque_real`,
         COUNT(DISTINCT `vl_preco_unitario`) AS `vl_preco_unitario`,
         COUNT(DISTINCT `dt_ingestao`) AS `dt_ingestao`
  FROM d GROUP BY `num_ledger_material`, `num_item_ledger_material`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(97,
    'cod_material', MAX(`cod_material`),
    'cod_area_avaliacao', MAX(`cod_area_avaliacao`),
    'cod_referencia_documento', MAX(`cod_referencia_documento`),
    'tp_moeda_externa_perspectiva_avaliacao', MAX(`tp_moeda_externa_perspectiva_avaliacao`),
    'cod_ledger_contabilidade_geral', MAX(`cod_ledger_contabilidade_geral`),
    'cod_referencia_execucao_calculo_custos', MAX(`cod_referencia_execucao_calculo_custos`),
    'cod_principio_contabil', MAX(`cod_principio_contabil`),
    'num_calculo_custos_sem_estrutura_qtd', MAX(`num_calculo_custos_sem_estrutura_qtd`),
    'cod_ano_periodo_aaaappp_lancamento', MAX(`cod_ano_periodo_aaaappp_lancamento`),
    'cod_ano_periodo_origem_material_processo', MAX(`cod_ano_periodo_origem_material_processo`),
    'cod_ano_corrente_periodo_yyyyppp_momento', MAX(`cod_ano_corrente_periodo_yyyyppp_momento`),
    'ind_calculo_custos', MAX(`ind_calculo_custos`),
    'cod_operacao_empresarial', MAX(`cod_operacao_empresarial`),
    'tp_ledger_material_operacao', MAX(`tp_ledger_material_operacao`),
    'tp_item_documento_ledger_material', MAX(`tp_item_documento_ledger_material`),
    'tp_exibicao_dados_ledger_material', MAX(`tp_exibicao_dados_ledger_material`),
    'tp_processo_alternativa_suprimento_consumo', MAX(`tp_processo_alternativa_suprimento_consumo`),
    'cod_grupo_tipos_movimento_ledger_material', MAX(`cod_grupo_tipos_movimento_ledger_material`),
    'cod_alternativa_processo_suprimento', MAX(`cod_alternativa_processo_suprimento`),
    'cod_processo_producao', MAX(`cod_processo_producao`),
    'num_calculo_custos_quantidade_produzida', MAX(`num_calculo_custos_quantidade_produzida`),
    'qt_transacao_unidade_medida_avaliacao', MAX(`qt_transacao_unidade_medida_avaliacao`),
    'vl_estoque_operacional', MAX(`vl_estoque_operacional`),
    'vl_diferenca_preco', MAX(`vl_diferenca_preco`),
    'vl_desvio_taxa_cambio', MAX(`vl_desvio_taxa_cambio`),
    'vl_diferenca_transferencia', MAX(`vl_diferenca_transferencia`),
    'vl_estoque_alternativo_interno_periodico', MAX(`vl_estoque_alternativo_interno_periodico`),
    'qt_estoque_avaliado', MAX(`qt_estoque_avaliado`),
    'vl_estoque_avaliado_total_antes_transacao', MAX(`vl_estoque_avaliado_total_antes_transacao`),
    'vl_alternativo_estoque_avaliado_total_antes', MAX(`vl_alternativo_estoque_avaliado_total_antes`),
    'vl_preco_padrao_antes_transacao', MAX(`vl_preco_padrao_antes_transacao`),
    'qt_unidade_preco_antes_transacao', MAX(`qt_unidade_preco_antes_transacao`),
    'vl_venda_calculo_real_artigo', MAX(`vl_venda_calculo_real_artigo`),
    'qt_limitacao_preco', MAX(`qt_limitacao_preco`),
    'vl_lucro_intercompanhias', MAX(`vl_lucro_intercompanhias`),
    'vl_estoque_venda', MAX(`vl_estoque_venda`),
    'vl_coeficiente_equivalencia_producao_conjunta', MAX(`vl_coeficiente_equivalencia_producao_conjunta`),
    'vl_preco_medio_movel_antes_transacao', MAX(`vl_preco_medio_movel_antes_transacao`),
    'dh_registro_hora', MAX(`dh_registro_hora`),
    'qt_pedido_unidade_medida_pedido', MAX(`qt_pedido_unidade_medida_pedido`),
    'vl_entrada_fatura_ordem_producao', MAX(`vl_entrada_fatura_ordem_producao`),
    'vl_liquidacao_ordem_producao', MAX(`vl_liquidacao_ordem_producao`),
    'vl_estoque_operacional_centro_lucro', MAX(`vl_estoque_operacional_centro_lucro`),
    'qt_referencia_fatura_liq_ordem_producao', MAX(`qt_referencia_fatura_liq_ordem_producao`),
    'vl_diferenca_preco_centro_lucro', MAX(`vl_diferenca_preco_centro_lucro`),
    'sg_quantidade_avaliacao', MAX(`sg_quantidade_avaliacao`),
    'cod_moeda', MAX(`cod_moeda`),
    'ind_preco_desvio_taxa_cambio_relevantes', MAX(`ind_preco_desvio_taxa_cambio_relevantes`),
    'ind_desvios_sao_relevantes_logica_limitacao', MAX(`ind_desvios_sao_relevantes_logica_limitacao`),
    'ind_preco_interno_relevante_modificacao_centro', MAX(`ind_preco_interno_relevante_modificacao_centro`),
    'dt_lancamento_documento', MAX(`dt_lancamento_documento`),
    'num_ciclo_execucao', MAX(`num_ciclo_execucao`),
    'num_ordem', MAX(`num_ordem`),
    'num_item_ordem', MAX(`num_item_ordem`),
    'num_ledger_material_modificacao_preco_ano', MAX(`num_ledger_material_modificacao_preco_ano`),
    'cod_operacao_referencia', MAX(`cod_operacao_referencia`),
    'cod_sistema_logico_origem', MAX(`cod_sistema_logico_origem`),
    'sg_unidades_organizacionais_referencia', MAX(`sg_unidades_organizacionais_referencia`),
    'num_documento_referencia', MAX(`num_documento_referencia`),
    'num_partida_individual_referencia', MAX(`num_partida_individual_referencia`),
    'num_grupo_partidas_individuais_referencia', MAX(`num_grupo_partidas_individuais_referencia`),
    'nm_usuario', MAX(`nm_usuario`),
    'cod_transacao', MAX(`cod_transacao`),
    'cod_operacao_ledger_material', MAX(`cod_operacao_ledger_material`),
    'tp_transacao_pedido_historico_ordem_producao', MAX(`tp_transacao_pedido_historico_ordem_producao`),
    'cod_grupo_origem_classe_custo', MAX(`cod_grupo_origem_classe_custo`),
    'cod_classe_custo_ordem', MAX(`cod_classe_custo_ordem`),
    'tp_movimento_administracao_estoques', MAX(`tp_movimento_administracao_estoques`),
    'cod_centro_ordem_producao_dados_historico', MAX(`cod_centro_ordem_producao_dados_historico`),
    'ind_estorno_apropriacao_ordem', MAX(`ind_estorno_apropriacao_ordem`),
    'sg_unidade_medida_ordem_producao', MAX(`sg_unidade_medida_ordem_producao`),
    'cod_periodo_apropriacao_custos_ordem', MAX(`cod_periodo_apropriacao_custos_ordem`),
    'ind_capitalizar_diferencas_cambio', MAX(`ind_capitalizar_diferencas_cambio`),
    'dt_envelhecimento_dados', MAX(`dt_envelhecimento_dados`),
    'cod_motivo_classificacao_contabil', MAX(`cod_motivo_classificacao_contabil`),
    'cod_motivo_modificacao', MAX(`cod_motivo_modificacao`),
    'cod_versao_migracao_dados', MAX(`cod_versao_migracao_dados`),
    'st_cobertura_estoque', MAX(`st_cobertura_estoque`),
    'cod_caso_fallback', MAX(`cod_caso_fallback`),
    'tp_preco_fallback', MAX(`tp_preco_fallback`),
    'cod_avaliacao_externa', MAX(`cod_avaliacao_externa`),
    'ind_conexao_interrompida', MAX(`ind_conexao_interrompida`),
    'cod_moeda_centro_lucro', MAX(`cod_moeda_centro_lucro`),
    'num_documento_compras', MAX(`num_documento_compras`),
    'num_item_documento_compras', MAX(`num_item_documento_compras`),
    'num_exercicio_periodo_atual', MAX(`num_exercicio_periodo_atual`),
    'num_documento_referencia_historico', MAX(`num_documento_referencia_historico`),
    'num_item_documento_referencia_historico', MAX(`num_item_documento_referencia_historico`),
    'tp_condicao', MAX(`tp_condicao`),
    'desc_categoria_ledger_material', MAX(`desc_categoria_ledger_material`),
    'dt_sistema', MAX(`dt_sistema`),
    'dh_carga', MAX(`dh_carga`),
    'dh_atualizacao', MAX(`dh_atualizacao`),
    'hash_conteudo', MAX(`hash_conteudo`),
    'vl_estoque_real', MAX(`vl_estoque_real`),
    'vl_preco_unitario', MAX(`vl_preco_unitario`),
    'dt_ingestao', MAX(`dt_ingestao`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Detecta coluna nunca carregada **neste centro**.

Uma coluna pode ter dado na tabela toda e estar vazia no centro 4128 — ou o contrário.
Por isso a varredura é feita sobre o recorte, não sobre a base completa.

In [0]:
-- PREENCHIMENTO DE TODAS AS COLUNAS NO CENARIO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'),
perf AS (
  SELECT stack(99,
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_area_avaliacao', 'string', COUNT_IF(`cod_area_avaliacao` IS NULL), COUNT_IF(`cod_area_avaliacao` IS NOT NULL AND lower(trim(`cod_area_avaliacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_area_avaliacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_referencia_documento', 'string', COUNT_IF(`cod_referencia_documento` IS NULL), COUNT_IF(`cod_referencia_documento` IS NOT NULL AND lower(trim(`cod_referencia_documento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_referencia_documento`) RLIKE '^0+([.,]0+)?$'),
    'tp_moeda_externa_perspectiva_avaliacao', 'string', COUNT_IF(`tp_moeda_externa_perspectiva_avaliacao` IS NULL), COUNT_IF(`tp_moeda_externa_perspectiva_avaliacao` IS NOT NULL AND lower(trim(`tp_moeda_externa_perspectiva_avaliacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_moeda_externa_perspectiva_avaliacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_ledger_contabilidade_geral', 'string', COUNT_IF(`cod_ledger_contabilidade_geral` IS NULL), COUNT_IF(`cod_ledger_contabilidade_geral` IS NOT NULL AND lower(trim(`cod_ledger_contabilidade_geral`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_ledger_contabilidade_geral`) RLIKE '^0+([.,]0+)?$'),
    'cod_referencia_execucao_calculo_custos', 'string', COUNT_IF(`cod_referencia_execucao_calculo_custos` IS NULL), COUNT_IF(`cod_referencia_execucao_calculo_custos` IS NOT NULL AND lower(trim(`cod_referencia_execucao_calculo_custos`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_referencia_execucao_calculo_custos`) RLIKE '^0+([.,]0+)?$'),
    'cod_principio_contabil', 'string', COUNT_IF(`cod_principio_contabil` IS NULL), COUNT_IF(`cod_principio_contabil` IS NOT NULL AND lower(trim(`cod_principio_contabil`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_principio_contabil`) RLIKE '^0+([.,]0+)?$'),
    'num_calculo_custos_sem_estrutura_qtd', 'string', COUNT_IF(`num_calculo_custos_sem_estrutura_qtd` IS NULL), COUNT_IF(`num_calculo_custos_sem_estrutura_qtd` IS NOT NULL AND lower(trim(`num_calculo_custos_sem_estrutura_qtd`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_calculo_custos_sem_estrutura_qtd`) RLIKE '^0+([.,]0+)?$'),
    'cod_ano_periodo_aaaappp_lancamento', 'string', COUNT_IF(`cod_ano_periodo_aaaappp_lancamento` IS NULL), COUNT_IF(`cod_ano_periodo_aaaappp_lancamento` IS NOT NULL AND lower(trim(`cod_ano_periodo_aaaappp_lancamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_ano_periodo_aaaappp_lancamento`) RLIKE '^0+([.,]0+)?$'),
    'cod_ano_periodo_origem_material_processo', 'string', COUNT_IF(`cod_ano_periodo_origem_material_processo` IS NULL), COUNT_IF(`cod_ano_periodo_origem_material_processo` IS NOT NULL AND lower(trim(`cod_ano_periodo_origem_material_processo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_ano_periodo_origem_material_processo`) RLIKE '^0+([.,]0+)?$'),
    'cod_ano_corrente_periodo_yyyyppp_momento', 'string', COUNT_IF(`cod_ano_corrente_periodo_yyyyppp_momento` IS NULL), COUNT_IF(`cod_ano_corrente_periodo_yyyyppp_momento` IS NOT NULL AND lower(trim(`cod_ano_corrente_periodo_yyyyppp_momento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_ano_corrente_periodo_yyyyppp_momento`) RLIKE '^0+([.,]0+)?$'),
    'ind_calculo_custos', 'boolean', COUNT_IF(`ind_calculo_custos` IS NULL), 0L, COUNT_IF(`ind_calculo_custos` = false),
    'cod_operacao_empresarial', 'string', COUNT_IF(`cod_operacao_empresarial` IS NULL), COUNT_IF(`cod_operacao_empresarial` IS NOT NULL AND lower(trim(`cod_operacao_empresarial`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_operacao_empresarial`) RLIKE '^0+([.,]0+)?$'),
    'tp_ledger_material_operacao', 'string', COUNT_IF(`tp_ledger_material_operacao` IS NULL), COUNT_IF(`tp_ledger_material_operacao` IS NOT NULL AND lower(trim(`tp_ledger_material_operacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_ledger_material_operacao`) RLIKE '^0+([.,]0+)?$'),
    'tp_item_documento_ledger_material', 'string', COUNT_IF(`tp_item_documento_ledger_material` IS NULL), COUNT_IF(`tp_item_documento_ledger_material` IS NOT NULL AND lower(trim(`tp_item_documento_ledger_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_item_documento_ledger_material`) RLIKE '^0+([.,]0+)?$'),
    'tp_exibicao_dados_ledger_material', 'string', COUNT_IF(`tp_exibicao_dados_ledger_material` IS NULL), COUNT_IF(`tp_exibicao_dados_ledger_material` IS NOT NULL AND lower(trim(`tp_exibicao_dados_ledger_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_exibicao_dados_ledger_material`) RLIKE '^0+([.,]0+)?$'),
    'tp_processo_alternativa_suprimento_consumo', 'string', COUNT_IF(`tp_processo_alternativa_suprimento_consumo` IS NULL), COUNT_IF(`tp_processo_alternativa_suprimento_consumo` IS NOT NULL AND lower(trim(`tp_processo_alternativa_suprimento_consumo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_processo_alternativa_suprimento_consumo`) RLIKE '^0+([.,]0+)?$'),
    'cod_grupo_tipos_movimento_ledger_material', 'string', COUNT_IF(`cod_grupo_tipos_movimento_ledger_material` IS NULL), COUNT_IF(`cod_grupo_tipos_movimento_ledger_material` IS NOT NULL AND lower(trim(`cod_grupo_tipos_movimento_ledger_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_grupo_tipos_movimento_ledger_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_alternativa_processo_suprimento', 'string', COUNT_IF(`cod_alternativa_processo_suprimento` IS NULL), COUNT_IF(`cod_alternativa_processo_suprimento` IS NOT NULL AND lower(trim(`cod_alternativa_processo_suprimento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_alternativa_processo_suprimento`) RLIKE '^0+([.,]0+)?$'),
    'cod_processo_producao', 'string', COUNT_IF(`cod_processo_producao` IS NULL), COUNT_IF(`cod_processo_producao` IS NOT NULL AND lower(trim(`cod_processo_producao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_processo_producao`) RLIKE '^0+([.,]0+)?$'),
    'num_calculo_custos_quantidade_produzida', 'string', COUNT_IF(`num_calculo_custos_quantidade_produzida` IS NULL), COUNT_IF(`num_calculo_custos_quantidade_produzida` IS NOT NULL AND lower(trim(`num_calculo_custos_quantidade_produzida`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_calculo_custos_quantidade_produzida`) RLIKE '^0+([.,]0+)?$'),
    'qt_transacao_unidade_medida_avaliacao', 'decimal(23,3)', COUNT_IF(`qt_transacao_unidade_medida_avaliacao` IS NULL), 0L, COUNT_IF(`qt_transacao_unidade_medida_avaliacao` = 0),
    'vl_estoque_operacional', 'double', COUNT_IF(`vl_estoque_operacional` IS NULL), 0L, COUNT_IF(`vl_estoque_operacional` = 0),
    'vl_diferenca_preco', 'double', COUNT_IF(`vl_diferenca_preco` IS NULL), 0L, COUNT_IF(`vl_diferenca_preco` = 0),
    'vl_desvio_taxa_cambio', 'double', COUNT_IF(`vl_desvio_taxa_cambio` IS NULL), 0L, COUNT_IF(`vl_desvio_taxa_cambio` = 0),
    'vl_diferenca_transferencia', 'double', COUNT_IF(`vl_diferenca_transferencia` IS NULL), 0L, COUNT_IF(`vl_diferenca_transferencia` = 0),
    'vl_estoque_alternativo_interno_periodico', 'double', COUNT_IF(`vl_estoque_alternativo_interno_periodico` IS NULL), 0L, COUNT_IF(`vl_estoque_alternativo_interno_periodico` = 0),
    'qt_estoque_avaliado', 'decimal(23,3)', COUNT_IF(`qt_estoque_avaliado` IS NULL), 0L, COUNT_IF(`qt_estoque_avaliado` = 0),
    'vl_estoque_avaliado_total_antes_transacao', 'double', COUNT_IF(`vl_estoque_avaliado_total_antes_transacao` IS NULL), 0L, COUNT_IF(`vl_estoque_avaliado_total_antes_transacao` = 0),
    'vl_alternativo_estoque_avaliado_total_antes', 'double', COUNT_IF(`vl_alternativo_estoque_avaliado_total_antes` IS NULL), 0L, COUNT_IF(`vl_alternativo_estoque_avaliado_total_antes` = 0),
    'vl_preco_padrao_antes_transacao', 'double', COUNT_IF(`vl_preco_padrao_antes_transacao` IS NULL), 0L, COUNT_IF(`vl_preco_padrao_antes_transacao` = 0),
    'qt_unidade_preco_antes_transacao', 'decimal(5,0)', COUNT_IF(`qt_unidade_preco_antes_transacao` IS NULL), 0L, COUNT_IF(`qt_unidade_preco_antes_transacao` = 0),
    'vl_venda_calculo_real_artigo', 'double', COUNT_IF(`vl_venda_calculo_real_artigo` IS NULL), 0L, COUNT_IF(`vl_venda_calculo_real_artigo` = 0),
    'qt_limitacao_preco', 'decimal(23,3)', COUNT_IF(`qt_limitacao_preco` IS NULL), 0L, COUNT_IF(`qt_limitacao_preco` = 0),
    'vl_lucro_intercompanhias', 'double', COUNT_IF(`vl_lucro_intercompanhias` IS NULL), 0L, COUNT_IF(`vl_lucro_intercompanhias` = 0),
    'vl_estoque_venda', 'double', COUNT_IF(`vl_estoque_venda` IS NULL), 0L, COUNT_IF(`vl_estoque_venda` = 0),
    'vl_coeficiente_equivalencia_producao_conjunta', 'double', COUNT_IF(`vl_coeficiente_equivalencia_producao_conjunta` IS NULL), 0L, COUNT_IF(`vl_coeficiente_equivalencia_producao_conjunta` = 0),
    'vl_preco_medio_movel_antes_transacao', 'double', COUNT_IF(`vl_preco_medio_movel_antes_transacao` IS NULL), 0L, COUNT_IF(`vl_preco_medio_movel_antes_transacao` = 0),
    'dh_registro_hora', 'timestamp', COUNT_IF(`dh_registro_hora` IS NULL), 0L, 0L,
    'qt_pedido_unidade_medida_pedido', 'decimal(13,3)', COUNT_IF(`qt_pedido_unidade_medida_pedido` IS NULL), 0L, COUNT_IF(`qt_pedido_unidade_medida_pedido` = 0),
    'vl_entrada_fatura_ordem_producao', 'double', COUNT_IF(`vl_entrada_fatura_ordem_producao` IS NULL), 0L, COUNT_IF(`vl_entrada_fatura_ordem_producao` = 0),
    'vl_liquidacao_ordem_producao', 'double', COUNT_IF(`vl_liquidacao_ordem_producao` IS NULL), 0L, COUNT_IF(`vl_liquidacao_ordem_producao` = 0),
    'vl_estoque_operacional_centro_lucro', 'double', COUNT_IF(`vl_estoque_operacional_centro_lucro` IS NULL), 0L, COUNT_IF(`vl_estoque_operacional_centro_lucro` = 0),
    'qt_referencia_fatura_liq_ordem_producao', 'decimal(13,3)', COUNT_IF(`qt_referencia_fatura_liq_ordem_producao` IS NULL), 0L, COUNT_IF(`qt_referencia_fatura_liq_ordem_producao` = 0),
    'vl_diferenca_preco_centro_lucro', 'double', COUNT_IF(`vl_diferenca_preco_centro_lucro` IS NULL), 0L, COUNT_IF(`vl_diferenca_preco_centro_lucro` = 0),
    'sg_quantidade_avaliacao', 'string', COUNT_IF(`sg_quantidade_avaliacao` IS NULL), COUNT_IF(`sg_quantidade_avaliacao` IS NOT NULL AND lower(trim(`sg_quantidade_avaliacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_quantidade_avaliacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_moeda', 'string', COUNT_IF(`cod_moeda` IS NULL), COUNT_IF(`cod_moeda` IS NOT NULL AND lower(trim(`cod_moeda`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_moeda`) RLIKE '^0+([.,]0+)?$'),
    'ind_preco_desvio_taxa_cambio_relevantes', 'boolean', COUNT_IF(`ind_preco_desvio_taxa_cambio_relevantes` IS NULL), 0L, COUNT_IF(`ind_preco_desvio_taxa_cambio_relevantes` = false),
    'ind_desvios_sao_relevantes_logica_limitacao', 'boolean', COUNT_IF(`ind_desvios_sao_relevantes_logica_limitacao` IS NULL), 0L, COUNT_IF(`ind_desvios_sao_relevantes_logica_limitacao` = false),
    'ind_preco_interno_relevante_modificacao_centro', 'boolean', COUNT_IF(`ind_preco_interno_relevante_modificacao_centro` IS NULL), 0L, COUNT_IF(`ind_preco_interno_relevante_modificacao_centro` = false),
    'dt_lancamento_documento', 'date', COUNT_IF(`dt_lancamento_documento` IS NULL), 0L, 0L,
    'num_ciclo_execucao', 'string', COUNT_IF(`num_ciclo_execucao` IS NULL), COUNT_IF(`num_ciclo_execucao` IS NOT NULL AND lower(trim(`num_ciclo_execucao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_ciclo_execucao`) RLIKE '^0+([.,]0+)?$'),
    'num_ordem', 'string', COUNT_IF(`num_ordem` IS NULL), COUNT_IF(`num_ordem` IS NOT NULL AND lower(trim(`num_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_ordem`) RLIKE '^0+([.,]0+)?$'),
    'num_item_ordem', 'string', COUNT_IF(`num_item_ordem` IS NULL), COUNT_IF(`num_item_ordem` IS NOT NULL AND lower(trim(`num_item_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_ordem`) RLIKE '^0+([.,]0+)?$'),
    'num_ledger_material', 'string', COUNT_IF(`num_ledger_material` IS NULL), COUNT_IF(`num_ledger_material` IS NOT NULL AND lower(trim(`num_ledger_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_ledger_material`) RLIKE '^0+([.,]0+)?$'),
    'num_ledger_material_modificacao_preco_ano', 'string', COUNT_IF(`num_ledger_material_modificacao_preco_ano` IS NULL), COUNT_IF(`num_ledger_material_modificacao_preco_ano` IS NOT NULL AND lower(trim(`num_ledger_material_modificacao_preco_ano`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_ledger_material_modificacao_preco_ano`) RLIKE '^0+([.,]0+)?$'),
    'num_item_ledger_material', 'string', COUNT_IF(`num_item_ledger_material` IS NULL), COUNT_IF(`num_item_ledger_material` IS NOT NULL AND lower(trim(`num_item_ledger_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_ledger_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_operacao_referencia', 'string', COUNT_IF(`cod_operacao_referencia` IS NULL), COUNT_IF(`cod_operacao_referencia` IS NOT NULL AND lower(trim(`cod_operacao_referencia`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_operacao_referencia`) RLIKE '^0+([.,]0+)?$'),
    'cod_sistema_logico_origem', 'string', COUNT_IF(`cod_sistema_logico_origem` IS NULL), COUNT_IF(`cod_sistema_logico_origem` IS NOT NULL AND lower(trim(`cod_sistema_logico_origem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_sistema_logico_origem`) RLIKE '^0+([.,]0+)?$'),
    'sg_unidades_organizacionais_referencia', 'string', COUNT_IF(`sg_unidades_organizacionais_referencia` IS NULL), COUNT_IF(`sg_unidades_organizacionais_referencia` IS NOT NULL AND lower(trim(`sg_unidades_organizacionais_referencia`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_unidades_organizacionais_referencia`) RLIKE '^0+([.,]0+)?$'),
    'num_documento_referencia', 'string', COUNT_IF(`num_documento_referencia` IS NULL), COUNT_IF(`num_documento_referencia` IS NOT NULL AND lower(trim(`num_documento_referencia`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_documento_referencia`) RLIKE '^0+([.,]0+)?$'),
    'num_partida_individual_referencia', 'string', COUNT_IF(`num_partida_individual_referencia` IS NULL), COUNT_IF(`num_partida_individual_referencia` IS NOT NULL AND lower(trim(`num_partida_individual_referencia`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_partida_individual_referencia`) RLIKE '^0+([.,]0+)?$'),
    'num_grupo_partidas_individuais_referencia', 'string', COUNT_IF(`num_grupo_partidas_individuais_referencia` IS NULL), COUNT_IF(`num_grupo_partidas_individuais_referencia` IS NOT NULL AND lower(trim(`num_grupo_partidas_individuais_referencia`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_grupo_partidas_individuais_referencia`) RLIKE '^0+([.,]0+)?$'),
    'nm_usuario', 'string', COUNT_IF(`nm_usuario` IS NULL), COUNT_IF(`nm_usuario` IS NOT NULL AND lower(trim(`nm_usuario`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_usuario`) RLIKE '^0+([.,]0+)?$'),
    'cod_transacao', 'string', COUNT_IF(`cod_transacao` IS NULL), COUNT_IF(`cod_transacao` IS NOT NULL AND lower(trim(`cod_transacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_transacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_operacao_ledger_material', 'string', COUNT_IF(`cod_operacao_ledger_material` IS NULL), COUNT_IF(`cod_operacao_ledger_material` IS NOT NULL AND lower(trim(`cod_operacao_ledger_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_operacao_ledger_material`) RLIKE '^0+([.,]0+)?$'),
    'tp_transacao_pedido_historico_ordem_producao', 'string', COUNT_IF(`tp_transacao_pedido_historico_ordem_producao` IS NULL), COUNT_IF(`tp_transacao_pedido_historico_ordem_producao` IS NOT NULL AND lower(trim(`tp_transacao_pedido_historico_ordem_producao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_transacao_pedido_historico_ordem_producao`) RLIKE '^0+([.,]0+)?$'),
    'cod_grupo_origem_classe_custo', 'string', COUNT_IF(`cod_grupo_origem_classe_custo` IS NULL), COUNT_IF(`cod_grupo_origem_classe_custo` IS NOT NULL AND lower(trim(`cod_grupo_origem_classe_custo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_grupo_origem_classe_custo`) RLIKE '^0+([.,]0+)?$'),
    'cod_classe_custo_ordem', 'string', COUNT_IF(`cod_classe_custo_ordem` IS NULL), COUNT_IF(`cod_classe_custo_ordem` IS NOT NULL AND lower(trim(`cod_classe_custo_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_classe_custo_ordem`) RLIKE '^0+([.,]0+)?$'),
    'tp_movimento_administracao_estoques', 'string', COUNT_IF(`tp_movimento_administracao_estoques` IS NULL), COUNT_IF(`tp_movimento_administracao_estoques` IS NOT NULL AND lower(trim(`tp_movimento_administracao_estoques`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_movimento_administracao_estoques`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_ordem_producao_dados_historico', 'string', COUNT_IF(`cod_centro_ordem_producao_dados_historico` IS NULL), COUNT_IF(`cod_centro_ordem_producao_dados_historico` IS NOT NULL AND lower(trim(`cod_centro_ordem_producao_dados_historico`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_ordem_producao_dados_historico`) RLIKE '^0+([.,]0+)?$'),
    'ind_estorno_apropriacao_ordem', 'boolean', COUNT_IF(`ind_estorno_apropriacao_ordem` IS NULL), 0L, COUNT_IF(`ind_estorno_apropriacao_ordem` = false),
    'sg_unidade_medida_ordem_producao', 'string', COUNT_IF(`sg_unidade_medida_ordem_producao` IS NULL), COUNT_IF(`sg_unidade_medida_ordem_producao` IS NOT NULL AND lower(trim(`sg_unidade_medida_ordem_producao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_unidade_medida_ordem_producao`) RLIKE '^0+([.,]0+)?$'),
    'cod_periodo_apropriacao_custos_ordem', 'string', COUNT_IF(`cod_periodo_apropriacao_custos_ordem` IS NULL), COUNT_IF(`cod_periodo_apropriacao_custos_ordem` IS NOT NULL AND lower(trim(`cod_periodo_apropriacao_custos_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_periodo_apropriacao_custos_ordem`) RLIKE '^0+([.,]0+)?$'),
    'ind_capitalizar_diferencas_cambio', 'boolean', COUNT_IF(`ind_capitalizar_diferencas_cambio` IS NULL), 0L, COUNT_IF(`ind_capitalizar_diferencas_cambio` = false),
    'dt_envelhecimento_dados', 'date', COUNT_IF(`dt_envelhecimento_dados` IS NULL), 0L, 0L,
    'cod_motivo_classificacao_contabil', 'string', COUNT_IF(`cod_motivo_classificacao_contabil` IS NULL), COUNT_IF(`cod_motivo_classificacao_contabil` IS NOT NULL AND lower(trim(`cod_motivo_classificacao_contabil`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_motivo_classificacao_contabil`) RLIKE '^0+([.,]0+)?$'),
    'cod_motivo_modificacao', 'string', COUNT_IF(`cod_motivo_modificacao` IS NULL), COUNT_IF(`cod_motivo_modificacao` IS NOT NULL AND lower(trim(`cod_motivo_modificacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_motivo_modificacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_versao_migracao_dados', 'string', COUNT_IF(`cod_versao_migracao_dados` IS NULL), COUNT_IF(`cod_versao_migracao_dados` IS NOT NULL AND lower(trim(`cod_versao_migracao_dados`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_versao_migracao_dados`) RLIKE '^0+([.,]0+)?$'),
    'st_cobertura_estoque', 'string', COUNT_IF(`st_cobertura_estoque` IS NULL), COUNT_IF(`st_cobertura_estoque` IS NOT NULL AND lower(trim(`st_cobertura_estoque`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`st_cobertura_estoque`) RLIKE '^0+([.,]0+)?$'),
    'cod_caso_fallback', 'string', COUNT_IF(`cod_caso_fallback` IS NULL), COUNT_IF(`cod_caso_fallback` IS NOT NULL AND lower(trim(`cod_caso_fallback`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_caso_fallback`) RLIKE '^0+([.,]0+)?$'),
    'tp_preco_fallback', 'string', COUNT_IF(`tp_preco_fallback` IS NULL), COUNT_IF(`tp_preco_fallback` IS NOT NULL AND lower(trim(`tp_preco_fallback`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_preco_fallback`) RLIKE '^0+([.,]0+)?$'),
    'cod_avaliacao_externa', 'string', COUNT_IF(`cod_avaliacao_externa` IS NULL), COUNT_IF(`cod_avaliacao_externa` IS NOT NULL AND lower(trim(`cod_avaliacao_externa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_avaliacao_externa`) RLIKE '^0+([.,]0+)?$'),
    'ind_conexao_interrompida', 'boolean', COUNT_IF(`ind_conexao_interrompida` IS NULL), 0L, COUNT_IF(`ind_conexao_interrompida` = false),
    'cod_moeda_centro_lucro', 'string', COUNT_IF(`cod_moeda_centro_lucro` IS NULL), COUNT_IF(`cod_moeda_centro_lucro` IS NOT NULL AND lower(trim(`cod_moeda_centro_lucro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_moeda_centro_lucro`) RLIKE '^0+([.,]0+)?$'),
    'num_documento_compras', 'string', COUNT_IF(`num_documento_compras` IS NULL), COUNT_IF(`num_documento_compras` IS NOT NULL AND lower(trim(`num_documento_compras`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_documento_compras`) RLIKE '^0+([.,]0+)?$'),
    'num_item_documento_compras', 'string', COUNT_IF(`num_item_documento_compras` IS NULL), COUNT_IF(`num_item_documento_compras` IS NOT NULL AND lower(trim(`num_item_documento_compras`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_documento_compras`) RLIKE '^0+([.,]0+)?$'),
    'num_exercicio_periodo_atual', 'string', COUNT_IF(`num_exercicio_periodo_atual` IS NULL), COUNT_IF(`num_exercicio_periodo_atual` IS NOT NULL AND lower(trim(`num_exercicio_periodo_atual`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_exercicio_periodo_atual`) RLIKE '^0+([.,]0+)?$'),
    'num_documento_referencia_historico', 'string', COUNT_IF(`num_documento_referencia_historico` IS NULL), COUNT_IF(`num_documento_referencia_historico` IS NOT NULL AND lower(trim(`num_documento_referencia_historico`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_documento_referencia_historico`) RLIKE '^0+([.,]0+)?$'),
    'num_item_documento_referencia_historico', 'string', COUNT_IF(`num_item_documento_referencia_historico` IS NULL), COUNT_IF(`num_item_documento_referencia_historico` IS NOT NULL AND lower(trim(`num_item_documento_referencia_historico`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_documento_referencia_historico`) RLIKE '^0+([.,]0+)?$'),
    'tp_condicao', 'string', COUNT_IF(`tp_condicao` IS NULL), COUNT_IF(`tp_condicao` IS NOT NULL AND lower(trim(`tp_condicao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_condicao`) RLIKE '^0+([.,]0+)?$'),
    'desc_categoria_ledger_material', 'string', COUNT_IF(`desc_categoria_ledger_material` IS NULL), COUNT_IF(`desc_categoria_ledger_material` IS NOT NULL AND lower(trim(`desc_categoria_ledger_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_categoria_ledger_material`) RLIKE '^0+([.,]0+)?$'),
    'dt_sistema', 'date', COUNT_IF(`dt_sistema` IS NULL), 0L, 0L,
    'dh_carga', 'timestamp', COUNT_IF(`dh_carga` IS NULL), 0L, 0L,
    'dh_atualizacao', 'timestamp', COUNT_IF(`dh_atualizacao` IS NULL), 0L, 0L,
    'hash_conteudo', 'bigint', COUNT_IF(`hash_conteudo` IS NULL), 0L, COUNT_IF(`hash_conteudo` = 0),
    'vl_estoque_real', 'double', COUNT_IF(`vl_estoque_real` IS NULL), 0L, COUNT_IF(`vl_estoque_real` = 0),
    'vl_preco_unitario', 'double', COUNT_IF(`vl_preco_unitario` IS NULL), 0L, COUNT_IF(`vl_preco_unitario` = 0),
    'dt_ingestao', 'date', COUNT_IF(`dt_ingestao` IS NULL), 0L, 0L
  ) AS (coluna, tipo, nulos, vazios, zeros_ou_false)
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros_ou_false,
       t.total - p.nulos - p.vazios - p.zeros_ou_false                               AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros_ou_false) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total                                               THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros_ou_false <= 0             THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros_ou_false) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END                                                      AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade no cenário

In [0]:
-- CARDINALIDADE NO CENARIO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'),
card AS (
  SELECT stack(99,
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'cod_area_avaliacao', 'string', approx_count_distinct(`cod_area_avaliacao`),
    'cod_referencia_documento', 'string', approx_count_distinct(`cod_referencia_documento`),
    'tp_moeda_externa_perspectiva_avaliacao', 'string', approx_count_distinct(`tp_moeda_externa_perspectiva_avaliacao`),
    'cod_ledger_contabilidade_geral', 'string', approx_count_distinct(`cod_ledger_contabilidade_geral`),
    'cod_referencia_execucao_calculo_custos', 'string', approx_count_distinct(`cod_referencia_execucao_calculo_custos`),
    'cod_principio_contabil', 'string', approx_count_distinct(`cod_principio_contabil`),
    'num_calculo_custos_sem_estrutura_qtd', 'string', approx_count_distinct(`num_calculo_custos_sem_estrutura_qtd`),
    'cod_ano_periodo_aaaappp_lancamento', 'string', approx_count_distinct(`cod_ano_periodo_aaaappp_lancamento`),
    'cod_ano_periodo_origem_material_processo', 'string', approx_count_distinct(`cod_ano_periodo_origem_material_processo`),
    'cod_ano_corrente_periodo_yyyyppp_momento', 'string', approx_count_distinct(`cod_ano_corrente_periodo_yyyyppp_momento`),
    'ind_calculo_custos', 'boolean', approx_count_distinct(`ind_calculo_custos`),
    'cod_operacao_empresarial', 'string', approx_count_distinct(`cod_operacao_empresarial`),
    'tp_ledger_material_operacao', 'string', approx_count_distinct(`tp_ledger_material_operacao`),
    'tp_item_documento_ledger_material', 'string', approx_count_distinct(`tp_item_documento_ledger_material`),
    'tp_exibicao_dados_ledger_material', 'string', approx_count_distinct(`tp_exibicao_dados_ledger_material`),
    'tp_processo_alternativa_suprimento_consumo', 'string', approx_count_distinct(`tp_processo_alternativa_suprimento_consumo`),
    'cod_grupo_tipos_movimento_ledger_material', 'string', approx_count_distinct(`cod_grupo_tipos_movimento_ledger_material`),
    'cod_alternativa_processo_suprimento', 'string', approx_count_distinct(`cod_alternativa_processo_suprimento`),
    'cod_processo_producao', 'string', approx_count_distinct(`cod_processo_producao`),
    'num_calculo_custos_quantidade_produzida', 'string', approx_count_distinct(`num_calculo_custos_quantidade_produzida`),
    'qt_transacao_unidade_medida_avaliacao', 'decimal(23,3)', approx_count_distinct(`qt_transacao_unidade_medida_avaliacao`),
    'vl_estoque_operacional', 'double', approx_count_distinct(`vl_estoque_operacional`),
    'vl_diferenca_preco', 'double', approx_count_distinct(`vl_diferenca_preco`),
    'vl_desvio_taxa_cambio', 'double', approx_count_distinct(`vl_desvio_taxa_cambio`),
    'vl_diferenca_transferencia', 'double', approx_count_distinct(`vl_diferenca_transferencia`),
    'vl_estoque_alternativo_interno_periodico', 'double', approx_count_distinct(`vl_estoque_alternativo_interno_periodico`),
    'qt_estoque_avaliado', 'decimal(23,3)', approx_count_distinct(`qt_estoque_avaliado`),
    'vl_estoque_avaliado_total_antes_transacao', 'double', approx_count_distinct(`vl_estoque_avaliado_total_antes_transacao`),
    'vl_alternativo_estoque_avaliado_total_antes', 'double', approx_count_distinct(`vl_alternativo_estoque_avaliado_total_antes`),
    'vl_preco_padrao_antes_transacao', 'double', approx_count_distinct(`vl_preco_padrao_antes_transacao`),
    'qt_unidade_preco_antes_transacao', 'decimal(5,0)', approx_count_distinct(`qt_unidade_preco_antes_transacao`),
    'vl_venda_calculo_real_artigo', 'double', approx_count_distinct(`vl_venda_calculo_real_artigo`),
    'qt_limitacao_preco', 'decimal(23,3)', approx_count_distinct(`qt_limitacao_preco`),
    'vl_lucro_intercompanhias', 'double', approx_count_distinct(`vl_lucro_intercompanhias`),
    'vl_estoque_venda', 'double', approx_count_distinct(`vl_estoque_venda`),
    'vl_coeficiente_equivalencia_producao_conjunta', 'double', approx_count_distinct(`vl_coeficiente_equivalencia_producao_conjunta`),
    'vl_preco_medio_movel_antes_transacao', 'double', approx_count_distinct(`vl_preco_medio_movel_antes_transacao`),
    'dh_registro_hora', 'timestamp', approx_count_distinct(`dh_registro_hora`),
    'qt_pedido_unidade_medida_pedido', 'decimal(13,3)', approx_count_distinct(`qt_pedido_unidade_medida_pedido`),
    'vl_entrada_fatura_ordem_producao', 'double', approx_count_distinct(`vl_entrada_fatura_ordem_producao`),
    'vl_liquidacao_ordem_producao', 'double', approx_count_distinct(`vl_liquidacao_ordem_producao`),
    'vl_estoque_operacional_centro_lucro', 'double', approx_count_distinct(`vl_estoque_operacional_centro_lucro`),
    'qt_referencia_fatura_liq_ordem_producao', 'decimal(13,3)', approx_count_distinct(`qt_referencia_fatura_liq_ordem_producao`),
    'vl_diferenca_preco_centro_lucro', 'double', approx_count_distinct(`vl_diferenca_preco_centro_lucro`),
    'sg_quantidade_avaliacao', 'string', approx_count_distinct(`sg_quantidade_avaliacao`),
    'cod_moeda', 'string', approx_count_distinct(`cod_moeda`),
    'ind_preco_desvio_taxa_cambio_relevantes', 'boolean', approx_count_distinct(`ind_preco_desvio_taxa_cambio_relevantes`),
    'ind_desvios_sao_relevantes_logica_limitacao', 'boolean', approx_count_distinct(`ind_desvios_sao_relevantes_logica_limitacao`),
    'ind_preco_interno_relevante_modificacao_centro', 'boolean', approx_count_distinct(`ind_preco_interno_relevante_modificacao_centro`),
    'dt_lancamento_documento', 'date', approx_count_distinct(`dt_lancamento_documento`),
    'num_ciclo_execucao', 'string', approx_count_distinct(`num_ciclo_execucao`),
    'num_ordem', 'string', approx_count_distinct(`num_ordem`),
    'num_item_ordem', 'string', approx_count_distinct(`num_item_ordem`),
    'num_ledger_material', 'string', approx_count_distinct(`num_ledger_material`),
    'num_ledger_material_modificacao_preco_ano', 'string', approx_count_distinct(`num_ledger_material_modificacao_preco_ano`),
    'num_item_ledger_material', 'string', approx_count_distinct(`num_item_ledger_material`),
    'cod_operacao_referencia', 'string', approx_count_distinct(`cod_operacao_referencia`),
    'cod_sistema_logico_origem', 'string', approx_count_distinct(`cod_sistema_logico_origem`),
    'sg_unidades_organizacionais_referencia', 'string', approx_count_distinct(`sg_unidades_organizacionais_referencia`),
    'num_documento_referencia', 'string', approx_count_distinct(`num_documento_referencia`),
    'num_partida_individual_referencia', 'string', approx_count_distinct(`num_partida_individual_referencia`),
    'num_grupo_partidas_individuais_referencia', 'string', approx_count_distinct(`num_grupo_partidas_individuais_referencia`),
    'nm_usuario', 'string', approx_count_distinct(`nm_usuario`),
    'cod_transacao', 'string', approx_count_distinct(`cod_transacao`),
    'cod_operacao_ledger_material', 'string', approx_count_distinct(`cod_operacao_ledger_material`),
    'tp_transacao_pedido_historico_ordem_producao', 'string', approx_count_distinct(`tp_transacao_pedido_historico_ordem_producao`),
    'cod_grupo_origem_classe_custo', 'string', approx_count_distinct(`cod_grupo_origem_classe_custo`),
    'cod_classe_custo_ordem', 'string', approx_count_distinct(`cod_classe_custo_ordem`),
    'tp_movimento_administracao_estoques', 'string', approx_count_distinct(`tp_movimento_administracao_estoques`),
    'cod_centro_ordem_producao_dados_historico', 'string', approx_count_distinct(`cod_centro_ordem_producao_dados_historico`),
    'ind_estorno_apropriacao_ordem', 'boolean', approx_count_distinct(`ind_estorno_apropriacao_ordem`),
    'sg_unidade_medida_ordem_producao', 'string', approx_count_distinct(`sg_unidade_medida_ordem_producao`),
    'cod_periodo_apropriacao_custos_ordem', 'string', approx_count_distinct(`cod_periodo_apropriacao_custos_ordem`),
    'ind_capitalizar_diferencas_cambio', 'boolean', approx_count_distinct(`ind_capitalizar_diferencas_cambio`),
    'dt_envelhecimento_dados', 'date', approx_count_distinct(`dt_envelhecimento_dados`),
    'cod_motivo_classificacao_contabil', 'string', approx_count_distinct(`cod_motivo_classificacao_contabil`),
    'cod_motivo_modificacao', 'string', approx_count_distinct(`cod_motivo_modificacao`),
    'cod_versao_migracao_dados', 'string', approx_count_distinct(`cod_versao_migracao_dados`),
    'st_cobertura_estoque', 'string', approx_count_distinct(`st_cobertura_estoque`),
    'cod_caso_fallback', 'string', approx_count_distinct(`cod_caso_fallback`),
    'tp_preco_fallback', 'string', approx_count_distinct(`tp_preco_fallback`),
    'cod_avaliacao_externa', 'string', approx_count_distinct(`cod_avaliacao_externa`),
    'ind_conexao_interrompida', 'boolean', approx_count_distinct(`ind_conexao_interrompida`),
    'cod_moeda_centro_lucro', 'string', approx_count_distinct(`cod_moeda_centro_lucro`),
    'num_documento_compras', 'string', approx_count_distinct(`num_documento_compras`),
    'num_item_documento_compras', 'string', approx_count_distinct(`num_item_documento_compras`),
    'num_exercicio_periodo_atual', 'string', approx_count_distinct(`num_exercicio_periodo_atual`),
    'num_documento_referencia_historico', 'string', approx_count_distinct(`num_documento_referencia_historico`),
    'num_item_documento_referencia_historico', 'string', approx_count_distinct(`num_item_documento_referencia_historico`),
    'tp_condicao', 'string', approx_count_distinct(`tp_condicao`),
    'desc_categoria_ledger_material', 'string', approx_count_distinct(`desc_categoria_ledger_material`),
    'dt_sistema', 'date', approx_count_distinct(`dt_sistema`),
    'dh_carga', 'timestamp', approx_count_distinct(`dh_carga`),
    'dh_atualizacao', 'timestamp', approx_count_distinct(`dh_atualizacao`),
    'hash_conteudo', 'bigint', approx_count_distinct(`hash_conteudo`),
    'vl_estoque_real', 'double', approx_count_distinct(`vl_estoque_real`),
    'vl_preco_unitario', 'double', approx_count_distinct(`vl_preco_unitario`),
    'dt_ingestao', 'date', approx_count_distinct(`dt_ingestao`)
  ) AS (coluna, tipo, distintos)
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1             THEN '1. CONSTANTE'
            WHEN c.distintos <= 3             THEN '2. cardinalidade muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador'
            ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada uma, dentro do cenário.

In [0]:
-- DOMINIO DAS CATEGORICAS NO CENARIO
(SELECT 'cod_area_avaliacao' AS coluna, CAST(`cod_area_avaliacao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128' GROUP BY `cod_area_avaliacao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_moeda' AS coluna, CAST(`cod_moeda` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128' GROUP BY `cod_moeda` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_ano_periodo_aaaappp_lancamento' AS coluna, CAST(`cod_ano_periodo_aaaappp_lancamento` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128' GROUP BY `cod_ano_periodo_aaaappp_lancamento` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_ledger_material_operacao' AS coluna, CAST(`tp_ledger_material_operacao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128' GROUP BY `tp_ledger_material_operacao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_item_documento_ledger_material' AS coluna, CAST(`tp_item_documento_ledger_material` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128' GROUP BY `tp_item_documento_ledger_material` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_principio_contabil' AS coluna, CAST(`cod_principio_contabil` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128' GROUP BY `cod_principio_contabil` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_movimento_administracao_estoques' AS coluna, CAST(`tp_movimento_administracao_estoques` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128' GROUP BY `tp_movimento_administracao_estoques` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'st_cobertura_estoque' AS coluna, CAST(`st_cobertura_estoque` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128' GROUP BY `st_cobertura_estoque` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_transacao' AS coluna, CAST(`cod_transacao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128' GROUP BY `cod_transacao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'desc_categoria_ledger_material' AS coluna, CAST(`desc_categoria_ledger_material` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128' GROUP BY `desc_categoria_ledger_material` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'sg_quantidade_avaliacao' AS coluna, CAST(`sg_quantidade_avaliacao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128' GROUP BY `sg_quantidade_avaliacao` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 9. Flags booleanas

> **Atenção na comparação:** o SAP exporta indicadores como `X` / vazio;
> o Datalake grava `true` / `false`. É conversão de tipo, não divergência de valor.

In [0]:
-- FLAGS BOOLEANAS
SELECT 'ind_calculo_custos' AS flag,
       COUNT_IF(`ind_calculo_custos` = true)  AS verdadeiro,
       COUNT_IF(`ind_calculo_custos` = false) AS falso,
       COUNT_IF(`ind_calculo_custos` IS NULL) AS nulo,
       ROUND(100.0 * COUNT_IF(`ind_calculo_custos` = true) / COUNT(*), 2) AS pct_verdadeiro
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
UNION ALL
SELECT 'ind_preco_desvio_taxa_cambio_relevantes' AS flag,
       COUNT_IF(`ind_preco_desvio_taxa_cambio_relevantes` = true)  AS verdadeiro,
       COUNT_IF(`ind_preco_desvio_taxa_cambio_relevantes` = false) AS falso,
       COUNT_IF(`ind_preco_desvio_taxa_cambio_relevantes` IS NULL) AS nulo,
       ROUND(100.0 * COUNT_IF(`ind_preco_desvio_taxa_cambio_relevantes` = true) / COUNT(*), 2) AS pct_verdadeiro
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
UNION ALL
SELECT 'ind_desvios_sao_relevantes_logica_limitacao' AS flag,
       COUNT_IF(`ind_desvios_sao_relevantes_logica_limitacao` = true)  AS verdadeiro,
       COUNT_IF(`ind_desvios_sao_relevantes_logica_limitacao` = false) AS falso,
       COUNT_IF(`ind_desvios_sao_relevantes_logica_limitacao` IS NULL) AS nulo,
       ROUND(100.0 * COUNT_IF(`ind_desvios_sao_relevantes_logica_limitacao` = true) / COUNT(*), 2) AS pct_verdadeiro
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
UNION ALL
SELECT 'ind_preco_interno_relevante_modificacao_centro' AS flag,
       COUNT_IF(`ind_preco_interno_relevante_modificacao_centro` = true)  AS verdadeiro,
       COUNT_IF(`ind_preco_interno_relevante_modificacao_centro` = false) AS falso,
       COUNT_IF(`ind_preco_interno_relevante_modificacao_centro` IS NULL) AS nulo,
       ROUND(100.0 * COUNT_IF(`ind_preco_interno_relevante_modificacao_centro` = true) / COUNT(*), 2) AS pct_verdadeiro
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
UNION ALL
SELECT 'ind_estorno_apropriacao_ordem' AS flag,
       COUNT_IF(`ind_estorno_apropriacao_ordem` = true)  AS verdadeiro,
       COUNT_IF(`ind_estorno_apropriacao_ordem` = false) AS falso,
       COUNT_IF(`ind_estorno_apropriacao_ordem` IS NULL) AS nulo,
       ROUND(100.0 * COUNT_IF(`ind_estorno_apropriacao_ordem` = true) / COUNT(*), 2) AS pct_verdadeiro
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
UNION ALL
SELECT 'ind_capitalizar_diferencas_cambio' AS flag,
       COUNT_IF(`ind_capitalizar_diferencas_cambio` = true)  AS verdadeiro,
       COUNT_IF(`ind_capitalizar_diferencas_cambio` = false) AS falso,
       COUNT_IF(`ind_capitalizar_diferencas_cambio` IS NULL) AS nulo,
       ROUND(100.0 * COUNT_IF(`ind_capitalizar_diferencas_cambio` = true) / COUNT(*), 2) AS pct_verdadeiro
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
UNION ALL
SELECT 'ind_conexao_interrompida' AS flag,
       COUNT_IF(`ind_conexao_interrompida` = true)  AS verdadeiro,
       COUNT_IF(`ind_conexao_interrompida` = false) AS falso,
       COUNT_IF(`ind_conexao_interrompida` IS NULL) AS nulo,
       ROUND(100.0 * COUNT_IF(`ind_conexao_interrompida` = true) / COUNT(*), 2) AS pct_verdadeiro
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
ORDER BY pct_verdadeiro DESC;

## 10. Perfil dos campos numéricos

A tabela tem **8 colunas `double`** entre as analisadas — exigem tolerância de 0,005 na comparação com o SAP.

In [0]:
-- PERFIL NUMERICO NO CENARIO
SELECT * FROM (
  SELECT stack(10,
    'qt_transacao_unidade_medida_avaliacao', 'decimal(23,3)', COUNT(`qt_transacao_unidade_medida_avaliacao`), CAST(MIN(`qt_transacao_unidade_medida_avaliacao`) AS DOUBLE), CAST(MAX(`qt_transacao_unidade_medida_avaliacao`) AS DOUBLE), CAST(AVG(`qt_transacao_unidade_medida_avaliacao`) AS DOUBLE), CAST(percentile_approx(`qt_transacao_unidade_medida_avaliacao`, 0.5) AS DOUBLE), COUNT_IF(`qt_transacao_unidade_medida_avaliacao` < 0), COUNT_IF(`qt_transacao_unidade_medida_avaliacao` = 0),
    'vl_estoque_operacional', 'double', COUNT(`vl_estoque_operacional`), CAST(MIN(`vl_estoque_operacional`) AS DOUBLE), CAST(MAX(`vl_estoque_operacional`) AS DOUBLE), CAST(AVG(`vl_estoque_operacional`) AS DOUBLE), CAST(percentile_approx(`vl_estoque_operacional`, 0.5) AS DOUBLE), COUNT_IF(`vl_estoque_operacional` < 0), COUNT_IF(`vl_estoque_operacional` = 0),
    'vl_diferenca_preco', 'double', COUNT(`vl_diferenca_preco`), CAST(MIN(`vl_diferenca_preco`) AS DOUBLE), CAST(MAX(`vl_diferenca_preco`) AS DOUBLE), CAST(AVG(`vl_diferenca_preco`) AS DOUBLE), CAST(percentile_approx(`vl_diferenca_preco`, 0.5) AS DOUBLE), COUNT_IF(`vl_diferenca_preco` < 0), COUNT_IF(`vl_diferenca_preco` = 0),
    'vl_desvio_taxa_cambio', 'double', COUNT(`vl_desvio_taxa_cambio`), CAST(MIN(`vl_desvio_taxa_cambio`) AS DOUBLE), CAST(MAX(`vl_desvio_taxa_cambio`) AS DOUBLE), CAST(AVG(`vl_desvio_taxa_cambio`) AS DOUBLE), CAST(percentile_approx(`vl_desvio_taxa_cambio`, 0.5) AS DOUBLE), COUNT_IF(`vl_desvio_taxa_cambio` < 0), COUNT_IF(`vl_desvio_taxa_cambio` = 0),
    'qt_estoque_avaliado', 'decimal(23,3)', COUNT(`qt_estoque_avaliado`), CAST(MIN(`qt_estoque_avaliado`) AS DOUBLE), CAST(MAX(`qt_estoque_avaliado`) AS DOUBLE), CAST(AVG(`qt_estoque_avaliado`) AS DOUBLE), CAST(percentile_approx(`qt_estoque_avaliado`, 0.5) AS DOUBLE), COUNT_IF(`qt_estoque_avaliado` < 0), COUNT_IF(`qt_estoque_avaliado` = 0),
    'vl_estoque_avaliado_total_antes_transacao', 'double', COUNT(`vl_estoque_avaliado_total_antes_transacao`), CAST(MIN(`vl_estoque_avaliado_total_antes_transacao`) AS DOUBLE), CAST(MAX(`vl_estoque_avaliado_total_antes_transacao`) AS DOUBLE), CAST(AVG(`vl_estoque_avaliado_total_antes_transacao`) AS DOUBLE), CAST(percentile_approx(`vl_estoque_avaliado_total_antes_transacao`, 0.5) AS DOUBLE), COUNT_IF(`vl_estoque_avaliado_total_antes_transacao` < 0), COUNT_IF(`vl_estoque_avaliado_total_antes_transacao` = 0),
    'vl_preco_padrao_antes_transacao', 'double', COUNT(`vl_preco_padrao_antes_transacao`), CAST(MIN(`vl_preco_padrao_antes_transacao`) AS DOUBLE), CAST(MAX(`vl_preco_padrao_antes_transacao`) AS DOUBLE), CAST(AVG(`vl_preco_padrao_antes_transacao`) AS DOUBLE), CAST(percentile_approx(`vl_preco_padrao_antes_transacao`, 0.5) AS DOUBLE), COUNT_IF(`vl_preco_padrao_antes_transacao` < 0), COUNT_IF(`vl_preco_padrao_antes_transacao` = 0),
    'vl_preco_medio_movel_antes_transacao', 'double', COUNT(`vl_preco_medio_movel_antes_transacao`), CAST(MIN(`vl_preco_medio_movel_antes_transacao`) AS DOUBLE), CAST(MAX(`vl_preco_medio_movel_antes_transacao`) AS DOUBLE), CAST(AVG(`vl_preco_medio_movel_antes_transacao`) AS DOUBLE), CAST(percentile_approx(`vl_preco_medio_movel_antes_transacao`, 0.5) AS DOUBLE), COUNT_IF(`vl_preco_medio_movel_antes_transacao` < 0), COUNT_IF(`vl_preco_medio_movel_antes_transacao` = 0),
    'vl_estoque_real', 'double', COUNT(`vl_estoque_real`), CAST(MIN(`vl_estoque_real`) AS DOUBLE), CAST(MAX(`vl_estoque_real`) AS DOUBLE), CAST(AVG(`vl_estoque_real`) AS DOUBLE), CAST(percentile_approx(`vl_estoque_real`, 0.5) AS DOUBLE), COUNT_IF(`vl_estoque_real` < 0), COUNT_IF(`vl_estoque_real` = 0),
    'vl_preco_unitario', 'double', COUNT(`vl_preco_unitario`), CAST(MIN(`vl_preco_unitario`) AS DOUBLE), CAST(MAX(`vl_preco_unitario`) AS DOUBLE), CAST(AVG(`vl_preco_unitario`) AS DOUBLE), CAST(percentile_approx(`vl_preco_unitario`, 0.5) AS DOUBLE), COUNT_IF(`vl_preco_unitario` < 0), COUNT_IF(`vl_preco_unitario` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros)
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
)
ORDER BY coluna;

## 11. Totais para conciliação com o SAP

**Use esta tabela para bater os totais contra o extrato do SAP.**

Some as mesmas colunas no Excel e compare. Divergência de total detecta registro
faltando ou duplicado — cobre o ponto cego da contagem de linhas.

In [0]:
-- TOTAIS PARA CONCILIACAO COM O SAP
SELECT coluna, total_numerico, total_arredondado, linhas_preenchidas
FROM (
  SELECT stack(10,
    'qt_transacao_unidade_medida_avaliacao', CAST(SUM(`qt_transacao_unidade_medida_avaliacao`) AS DOUBLE), CAST(ROUND(SUM(`qt_transacao_unidade_medida_avaliacao`), 2) AS STRING), COUNT(`qt_transacao_unidade_medida_avaliacao`),
    'vl_estoque_operacional', CAST(SUM(`vl_estoque_operacional`) AS DOUBLE), CAST(ROUND(SUM(`vl_estoque_operacional`), 2) AS STRING), COUNT(`vl_estoque_operacional`),
    'vl_diferenca_preco', CAST(SUM(`vl_diferenca_preco`) AS DOUBLE), CAST(ROUND(SUM(`vl_diferenca_preco`), 2) AS STRING), COUNT(`vl_diferenca_preco`),
    'vl_desvio_taxa_cambio', CAST(SUM(`vl_desvio_taxa_cambio`) AS DOUBLE), CAST(ROUND(SUM(`vl_desvio_taxa_cambio`), 2) AS STRING), COUNT(`vl_desvio_taxa_cambio`),
    'qt_estoque_avaliado', CAST(SUM(`qt_estoque_avaliado`) AS DOUBLE), CAST(ROUND(SUM(`qt_estoque_avaliado`), 2) AS STRING), COUNT(`qt_estoque_avaliado`),
    'vl_estoque_avaliado_total_antes_transacao', CAST(SUM(`vl_estoque_avaliado_total_antes_transacao`) AS DOUBLE), CAST(ROUND(SUM(`vl_estoque_avaliado_total_antes_transacao`), 2) AS STRING), COUNT(`vl_estoque_avaliado_total_antes_transacao`),
    'vl_preco_padrao_antes_transacao', CAST(SUM(`vl_preco_padrao_antes_transacao`) AS DOUBLE), CAST(ROUND(SUM(`vl_preco_padrao_antes_transacao`), 2) AS STRING), COUNT(`vl_preco_padrao_antes_transacao`),
    'vl_preco_medio_movel_antes_transacao', CAST(SUM(`vl_preco_medio_movel_antes_transacao`) AS DOUBLE), CAST(ROUND(SUM(`vl_preco_medio_movel_antes_transacao`), 2) AS STRING), COUNT(`vl_preco_medio_movel_antes_transacao`),
    'vl_estoque_real', CAST(SUM(`vl_estoque_real`) AS DOUBLE), CAST(ROUND(SUM(`vl_estoque_real`), 2) AS STRING), COUNT(`vl_estoque_real`),
    'vl_preco_unitario', CAST(SUM(`vl_preco_unitario`) AS DOUBLE), CAST(ROUND(SUM(`vl_preco_unitario`), 2) AS STRING), COUNT(`vl_preco_unitario`)
  ) AS (coluna, total_numerico, total_arredondado, linhas_preenchidas)
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
)
ORDER BY coluna;

## 12. Datas em tipo nativo

In [0]:
-- DATAS EM TIPO NATIVO
SELECT * FROM (
  SELECT stack(4,
    'dt_lancamento_documento', COUNT_IF(`dt_lancamento_documento` IS NULL), CAST(MIN(`dt_lancamento_documento`) AS STRING), CAST(MAX(`dt_lancamento_documento`) AS STRING), COUNT(DISTINCT `dt_lancamento_documento`), COUNT_IF(`dt_lancamento_documento` > current_date()),
    'dt_envelhecimento_dados', COUNT_IF(`dt_envelhecimento_dados` IS NULL), CAST(MIN(`dt_envelhecimento_dados`) AS STRING), CAST(MAX(`dt_envelhecimento_dados`) AS STRING), COUNT(DISTINCT `dt_envelhecimento_dados`), COUNT_IF(`dt_envelhecimento_dados` > current_date()),
    'dt_sistema', COUNT_IF(`dt_sistema` IS NULL), CAST(MIN(`dt_sistema`) AS STRING), CAST(MAX(`dt_sistema`) AS STRING), COUNT(DISTINCT `dt_sistema`), COUNT_IF(`dt_sistema` > current_date()),
    'dt_ingestao', COUNT_IF(`dt_ingestao` IS NULL), CAST(MIN(`dt_ingestao`) AS STRING), CAST(MAX(`dt_ingestao`) AS STRING), COUNT(DISTINCT `dt_ingestao`), COUNT_IF(`dt_ingestao` > current_date())
  ) AS (coluna, nulos, minimo, maximo, distintas, futuras)
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
)
ORDER BY coluna;

## 12.2 Timestamps

In [0]:
-- TIMESTAMPS
SELECT * FROM (
  SELECT stack(3,
    'dh_registro_hora', COUNT_IF(`dh_registro_hora` IS NULL), CAST(MIN(`dh_registro_hora`) AS STRING), CAST(MAX(`dh_registro_hora`) AS STRING), COUNT(DISTINCT `dh_registro_hora`),
    'dh_carga', COUNT_IF(`dh_carga` IS NULL), CAST(MIN(`dh_carga`) AS STRING), CAST(MAX(`dh_carga`) AS STRING), COUNT(DISTINCT `dh_carga`),
    'dh_atualizacao', COUNT_IF(`dh_atualizacao` IS NULL), CAST(MIN(`dh_atualizacao`) AS STRING), CAST(MAX(`dh_atualizacao`) AS STRING), COUNT(DISTINCT `dh_atualizacao`)
  ) AS (coluna, nulos, minimo, maximo, distintos)
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
)
ORDER BY coluna;

## 13. Códigos — zeros à esquerda e formato

**Armadilha:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá 0% de match.

In [0]:
-- CODIGOS: ZEROS A ESQUERDA E FORMATO
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes,
       CONCAT_WS(' | ',
         CASE WHEN tipo LIKE 'big%' OR tipo LIKE '%int%'
              THEN 'TIPO NUMERICO - zeros ja perdidos' END,
         CASE WHEN com_zeros_esq > 0 THEN 'normalizar antes do join' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END
       ) AS alertas
FROM (
  SELECT stack(7,
    'num_ledger_material', 'string', COUNT_IF(CAST(`num_ledger_material` AS STRING) IS NULL OR trim(CAST(`num_ledger_material` AS STRING)) = ''), MIN(length(trim(CAST(`num_ledger_material` AS STRING)))), MAX(length(trim(CAST(`num_ledger_material` AS STRING)))), COUNT_IF(trim(CAST(`num_ledger_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_ledger_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_ledger_material` AS STRING)), '^0+', '')),
    'num_item_ledger_material', 'string', COUNT_IF(CAST(`num_item_ledger_material` AS STRING) IS NULL OR trim(CAST(`num_item_ledger_material` AS STRING)) = ''), MIN(length(trim(CAST(`num_item_ledger_material` AS STRING)))), MAX(length(trim(CAST(`num_item_ledger_material` AS STRING)))), COUNT_IF(trim(CAST(`num_item_ledger_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_item_ledger_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_item_ledger_material` AS STRING)), '^0+', '')),
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'cod_area_avaliacao', 'string', COUNT_IF(CAST(`cod_area_avaliacao` AS STRING) IS NULL OR trim(CAST(`cod_area_avaliacao` AS STRING)) = ''), MIN(length(trim(CAST(`cod_area_avaliacao` AS STRING)))), MAX(length(trim(CAST(`cod_area_avaliacao` AS STRING)))), COUNT_IF(trim(CAST(`cod_area_avaliacao` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_area_avaliacao` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_area_avaliacao` AS STRING)), '^0+', '')),
    'num_ordem', 'string', COUNT_IF(CAST(`num_ordem` AS STRING) IS NULL OR trim(CAST(`num_ordem` AS STRING)) = ''), MIN(length(trim(CAST(`num_ordem` AS STRING)))), MAX(length(trim(CAST(`num_ordem` AS STRING)))), COUNT_IF(trim(CAST(`num_ordem` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_ordem` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_ordem` AS STRING)), '^0+', '')),
    'num_documento_compras', 'string', COUNT_IF(CAST(`num_documento_compras` AS STRING) IS NULL OR trim(CAST(`num_documento_compras` AS STRING)) = ''), MIN(length(trim(CAST(`num_documento_compras` AS STRING)))), MAX(length(trim(CAST(`num_documento_compras` AS STRING)))), COUNT_IF(trim(CAST(`num_documento_compras` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_documento_compras` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_documento_compras` AS STRING)), '^0+', '')),
    'num_item_documento_compras', 'string', COUNT_IF(CAST(`num_item_documento_compras` AS STRING) IS NULL OR trim(CAST(`num_item_documento_compras` AS STRING)) = ''), MIN(length(trim(CAST(`num_item_documento_compras` AS STRING)))), MAX(length(trim(CAST(`num_item_documento_compras` AS STRING)))), COUNT_IF(trim(CAST(`num_item_documento_compras` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_item_documento_compras` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_item_documento_compras` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
        distintos_bruto, distintos_sem_zeros)
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
)
ORDER BY coluna;

## 14. Chaves normalizadas para join com o SAP

Lista das chaves já **sem zeros à esquerda**, prontas para colar no Excel
e cruzar com o extrato do SAP via PROCV/ÍNDICE.

In [0]:
-- 14. CHAVES NORMALIZADAS
SELECT DISTINCT
       regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '') AS `cod_material_norm`,
       regexp_replace(trim(CAST(`cod_area_avaliacao` AS STRING)), '^0+', '') AS `cod_area_avaliacao_norm`,
       regexp_replace(trim(CAST(`cod_ano_periodo_aaaappp_lancamento` AS STRING)), '^0+', '') AS `cod_ano_periodo_aaaappp_lancamento_norm`
FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
ORDER BY 1, 2, 3;

## 15. Checksum de linha

Se `linhas` > `linhas_unicas`, existem registros 100% idênticos — duplicata real.

In [0]:
-- CHECKSUM DE LINHA
SELECT COUNT(*)                                            AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_area_avaliacao` AS STRING), ''), COALESCE(CAST(`cod_referencia_documento` AS STRING), ''), COALESCE(CAST(`tp_moeda_externa_perspectiva_avaliacao` AS STRING), ''), COALESCE(CAST(`cod_ledger_contabilidade_geral` AS STRING), ''), COALESCE(CAST(`cod_referencia_execucao_calculo_custos` AS STRING), ''), COALESCE(CAST(`cod_principio_contabil` AS STRING), ''), COALESCE(CAST(`num_calculo_custos_sem_estrutura_qtd` AS STRING), ''), COALESCE(CAST(`cod_ano_periodo_aaaappp_lancamento` AS STRING), ''), COALESCE(CAST(`cod_ano_periodo_origem_material_processo` AS STRING), ''), COALESCE(CAST(`cod_ano_corrente_periodo_yyyyppp_momento` AS STRING), ''), COALESCE(CAST(`ind_calculo_custos` AS STRING), ''), COALESCE(CAST(`cod_operacao_empresarial` AS STRING), ''), COALESCE(CAST(`tp_ledger_material_operacao` AS STRING), ''), COALESCE(CAST(`tp_item_documento_ledger_material` AS STRING), ''), COALESCE(CAST(`tp_exibicao_dados_ledger_material` AS STRING), ''), COALESCE(CAST(`tp_processo_alternativa_suprimento_consumo` AS STRING), ''), COALESCE(CAST(`cod_grupo_tipos_movimento_ledger_material` AS STRING), ''), COALESCE(CAST(`cod_alternativa_processo_suprimento` AS STRING), ''), COALESCE(CAST(`cod_processo_producao` AS STRING), ''), COALESCE(CAST(`num_calculo_custos_quantidade_produzida` AS STRING), ''), COALESCE(CAST(`qt_transacao_unidade_medida_avaliacao` AS STRING), ''), COALESCE(CAST(`vl_estoque_operacional` AS STRING), ''), COALESCE(CAST(`vl_diferenca_preco` AS STRING), ''), COALESCE(CAST(`vl_desvio_taxa_cambio` AS STRING), ''), COALESCE(CAST(`vl_diferenca_transferencia` AS STRING), ''), COALESCE(CAST(`vl_estoque_alternativo_interno_periodico` AS STRING), ''), COALESCE(CAST(`qt_estoque_avaliado` AS STRING), ''), COALESCE(CAST(`vl_estoque_avaliado_total_antes_transacao` AS STRING), ''), COALESCE(CAST(`vl_alternativo_estoque_avaliado_total_antes` AS STRING), ''), COALESCE(CAST(`vl_preco_padrao_antes_transacao` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco_antes_transacao` AS STRING), ''), COALESCE(CAST(`vl_venda_calculo_real_artigo` AS STRING), ''), COALESCE(CAST(`qt_limitacao_preco` AS STRING), ''), COALESCE(CAST(`vl_lucro_intercompanhias` AS STRING), ''), COALESCE(CAST(`vl_estoque_venda` AS STRING), ''), COALESCE(CAST(`vl_coeficiente_equivalencia_producao_conjunta` AS STRING), ''), COALESCE(CAST(`vl_preco_medio_movel_antes_transacao` AS STRING), ''), COALESCE(CAST(`dh_registro_hora` AS STRING), ''), COALESCE(CAST(`qt_pedido_unidade_medida_pedido` AS STRING), ''), COALESCE(CAST(`vl_entrada_fatura_ordem_producao` AS STRING), ''), COALESCE(CAST(`vl_liquidacao_ordem_producao` AS STRING), ''), COALESCE(CAST(`vl_estoque_operacional_centro_lucro` AS STRING), ''), COALESCE(CAST(`qt_referencia_fatura_liq_ordem_producao` AS STRING), ''), COALESCE(CAST(`vl_diferenca_preco_centro_lucro` AS STRING), ''), COALESCE(CAST(`sg_quantidade_avaliacao` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`ind_preco_desvio_taxa_cambio_relevantes` AS STRING), ''), COALESCE(CAST(`ind_desvios_sao_relevantes_logica_limitacao` AS STRING), ''), COALESCE(CAST(`ind_preco_interno_relevante_modificacao_centro` AS STRING), ''), COALESCE(CAST(`dt_lancamento_documento` AS STRING), ''), COALESCE(CAST(`num_ciclo_execucao` AS STRING), ''), COALESCE(CAST(`num_ordem` AS STRING), ''), COALESCE(CAST(`num_item_ordem` AS STRING), ''), COALESCE(CAST(`num_ledger_material` AS STRING), ''), COALESCE(CAST(`num_ledger_material_modificacao_preco_ano` AS STRING), ''), COALESCE(CAST(`num_item_ledger_material` AS STRING), ''), COALESCE(CAST(`cod_operacao_referencia` AS STRING), ''), COALESCE(CAST(`cod_sistema_logico_origem` AS STRING), ''), COALESCE(CAST(`sg_unidades_organizacionais_referencia` AS STRING), ''), COALESCE(CAST(`num_documento_referencia` AS STRING), ''), COALESCE(CAST(`num_partida_individual_referencia` AS STRING), ''), COALESCE(CAST(`num_grupo_partidas_individuais_referencia` AS STRING), ''), COALESCE(CAST(`nm_usuario` AS STRING), ''), COALESCE(CAST(`cod_transacao` AS STRING), ''), COALESCE(CAST(`cod_operacao_ledger_material` AS STRING), ''), COALESCE(CAST(`tp_transacao_pedido_historico_ordem_producao` AS STRING), ''), COALESCE(CAST(`cod_grupo_origem_classe_custo` AS STRING), ''), COALESCE(CAST(`cod_classe_custo_ordem` AS STRING), ''), COALESCE(CAST(`tp_movimento_administracao_estoques` AS STRING), ''), COALESCE(CAST(`cod_centro_ordem_producao_dados_historico` AS STRING), ''), COALESCE(CAST(`ind_estorno_apropriacao_ordem` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_ordem_producao` AS STRING), ''), COALESCE(CAST(`cod_periodo_apropriacao_custos_ordem` AS STRING), ''), COALESCE(CAST(`ind_capitalizar_diferencas_cambio` AS STRING), ''), COALESCE(CAST(`dt_envelhecimento_dados` AS STRING), ''), COALESCE(CAST(`cod_motivo_classificacao_contabil` AS STRING), ''), COALESCE(CAST(`cod_motivo_modificacao` AS STRING), ''), COALESCE(CAST(`cod_versao_migracao_dados` AS STRING), ''), COALESCE(CAST(`st_cobertura_estoque` AS STRING), ''), COALESCE(CAST(`cod_caso_fallback` AS STRING), ''), COALESCE(CAST(`tp_preco_fallback` AS STRING), ''), COALESCE(CAST(`cod_avaliacao_externa` AS STRING), ''), COALESCE(CAST(`ind_conexao_interrompida` AS STRING), ''), COALESCE(CAST(`cod_moeda_centro_lucro` AS STRING), ''), COALESCE(CAST(`num_documento_compras` AS STRING), ''), COALESCE(CAST(`num_item_documento_compras` AS STRING), ''), COALESCE(CAST(`num_exercicio_periodo_atual` AS STRING), ''), COALESCE(CAST(`num_documento_referencia_historico` AS STRING), ''), COALESCE(CAST(`num_item_documento_referencia_historico` AS STRING), ''), COALESCE(CAST(`tp_condicao` AS STRING), ''), COALESCE(CAST(`desc_categoria_ledger_material` AS STRING), ''), COALESCE(CAST(`dt_sistema` AS STRING), ''), COALESCE(CAST(`dh_carga` AS STRING), ''), COALESCE(CAST(`dh_atualizacao` AS STRING), ''), COALESCE(CAST(`hash_conteudo` AS STRING), ''), COALESCE(CAST(`vl_estoque_real` AS STRING), ''), COALESCE(CAST(`vl_preco_unitario` AS STRING), ''), COALESCE(CAST(`dt_ingestao` AS STRING), ''))))            AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_area_avaliacao` AS STRING), ''), COALESCE(CAST(`cod_referencia_documento` AS STRING), ''), COALESCE(CAST(`tp_moeda_externa_perspectiva_avaliacao` AS STRING), ''), COALESCE(CAST(`cod_ledger_contabilidade_geral` AS STRING), ''), COALESCE(CAST(`cod_referencia_execucao_calculo_custos` AS STRING), ''), COALESCE(CAST(`cod_principio_contabil` AS STRING), ''), COALESCE(CAST(`num_calculo_custos_sem_estrutura_qtd` AS STRING), ''), COALESCE(CAST(`cod_ano_periodo_aaaappp_lancamento` AS STRING), ''), COALESCE(CAST(`cod_ano_periodo_origem_material_processo` AS STRING), ''), COALESCE(CAST(`cod_ano_corrente_periodo_yyyyppp_momento` AS STRING), ''), COALESCE(CAST(`ind_calculo_custos` AS STRING), ''), COALESCE(CAST(`cod_operacao_empresarial` AS STRING), ''), COALESCE(CAST(`tp_ledger_material_operacao` AS STRING), ''), COALESCE(CAST(`tp_item_documento_ledger_material` AS STRING), ''), COALESCE(CAST(`tp_exibicao_dados_ledger_material` AS STRING), ''), COALESCE(CAST(`tp_processo_alternativa_suprimento_consumo` AS STRING), ''), COALESCE(CAST(`cod_grupo_tipos_movimento_ledger_material` AS STRING), ''), COALESCE(CAST(`cod_alternativa_processo_suprimento` AS STRING), ''), COALESCE(CAST(`cod_processo_producao` AS STRING), ''), COALESCE(CAST(`num_calculo_custos_quantidade_produzida` AS STRING), ''), COALESCE(CAST(`qt_transacao_unidade_medida_avaliacao` AS STRING), ''), COALESCE(CAST(`vl_estoque_operacional` AS STRING), ''), COALESCE(CAST(`vl_diferenca_preco` AS STRING), ''), COALESCE(CAST(`vl_desvio_taxa_cambio` AS STRING), ''), COALESCE(CAST(`vl_diferenca_transferencia` AS STRING), ''), COALESCE(CAST(`vl_estoque_alternativo_interno_periodico` AS STRING), ''), COALESCE(CAST(`qt_estoque_avaliado` AS STRING), ''), COALESCE(CAST(`vl_estoque_avaliado_total_antes_transacao` AS STRING), ''), COALESCE(CAST(`vl_alternativo_estoque_avaliado_total_antes` AS STRING), ''), COALESCE(CAST(`vl_preco_padrao_antes_transacao` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco_antes_transacao` AS STRING), ''), COALESCE(CAST(`vl_venda_calculo_real_artigo` AS STRING), ''), COALESCE(CAST(`qt_limitacao_preco` AS STRING), ''), COALESCE(CAST(`vl_lucro_intercompanhias` AS STRING), ''), COALESCE(CAST(`vl_estoque_venda` AS STRING), ''), COALESCE(CAST(`vl_coeficiente_equivalencia_producao_conjunta` AS STRING), ''), COALESCE(CAST(`vl_preco_medio_movel_antes_transacao` AS STRING), ''), COALESCE(CAST(`dh_registro_hora` AS STRING), ''), COALESCE(CAST(`qt_pedido_unidade_medida_pedido` AS STRING), ''), COALESCE(CAST(`vl_entrada_fatura_ordem_producao` AS STRING), ''), COALESCE(CAST(`vl_liquidacao_ordem_producao` AS STRING), ''), COALESCE(CAST(`vl_estoque_operacional_centro_lucro` AS STRING), ''), COALESCE(CAST(`qt_referencia_fatura_liq_ordem_producao` AS STRING), ''), COALESCE(CAST(`vl_diferenca_preco_centro_lucro` AS STRING), ''), COALESCE(CAST(`sg_quantidade_avaliacao` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`ind_preco_desvio_taxa_cambio_relevantes` AS STRING), ''), COALESCE(CAST(`ind_desvios_sao_relevantes_logica_limitacao` AS STRING), ''), COALESCE(CAST(`ind_preco_interno_relevante_modificacao_centro` AS STRING), ''), COALESCE(CAST(`dt_lancamento_documento` AS STRING), ''), COALESCE(CAST(`num_ciclo_execucao` AS STRING), ''), COALESCE(CAST(`num_ordem` AS STRING), ''), COALESCE(CAST(`num_item_ordem` AS STRING), ''), COALESCE(CAST(`num_ledger_material` AS STRING), ''), COALESCE(CAST(`num_ledger_material_modificacao_preco_ano` AS STRING), ''), COALESCE(CAST(`num_item_ledger_material` AS STRING), ''), COALESCE(CAST(`cod_operacao_referencia` AS STRING), ''), COALESCE(CAST(`cod_sistema_logico_origem` AS STRING), ''), COALESCE(CAST(`sg_unidades_organizacionais_referencia` AS STRING), ''), COALESCE(CAST(`num_documento_referencia` AS STRING), ''), COALESCE(CAST(`num_partida_individual_referencia` AS STRING), ''), COALESCE(CAST(`num_grupo_partidas_individuais_referencia` AS STRING), ''), COALESCE(CAST(`nm_usuario` AS STRING), ''), COALESCE(CAST(`cod_transacao` AS STRING), ''), COALESCE(CAST(`cod_operacao_ledger_material` AS STRING), ''), COALESCE(CAST(`tp_transacao_pedido_historico_ordem_producao` AS STRING), ''), COALESCE(CAST(`cod_grupo_origem_classe_custo` AS STRING), ''), COALESCE(CAST(`cod_classe_custo_ordem` AS STRING), ''), COALESCE(CAST(`tp_movimento_administracao_estoques` AS STRING), ''), COALESCE(CAST(`cod_centro_ordem_producao_dados_historico` AS STRING), ''), COALESCE(CAST(`ind_estorno_apropriacao_ordem` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_ordem_producao` AS STRING), ''), COALESCE(CAST(`cod_periodo_apropriacao_custos_ordem` AS STRING), ''), COALESCE(CAST(`ind_capitalizar_diferencas_cambio` AS STRING), ''), COALESCE(CAST(`dt_envelhecimento_dados` AS STRING), ''), COALESCE(CAST(`cod_motivo_classificacao_contabil` AS STRING), ''), COALESCE(CAST(`cod_motivo_modificacao` AS STRING), ''), COALESCE(CAST(`cod_versao_migracao_dados` AS STRING), ''), COALESCE(CAST(`st_cobertura_estoque` AS STRING), ''), COALESCE(CAST(`cod_caso_fallback` AS STRING), ''), COALESCE(CAST(`tp_preco_fallback` AS STRING), ''), COALESCE(CAST(`cod_avaliacao_externa` AS STRING), ''), COALESCE(CAST(`ind_conexao_interrompida` AS STRING), ''), COALESCE(CAST(`cod_moeda_centro_lucro` AS STRING), ''), COALESCE(CAST(`num_documento_compras` AS STRING), ''), COALESCE(CAST(`num_item_documento_compras` AS STRING), ''), COALESCE(CAST(`num_exercicio_periodo_atual` AS STRING), ''), COALESCE(CAST(`num_documento_referencia_historico` AS STRING), ''), COALESCE(CAST(`num_item_documento_referencia_historico` AS STRING), ''), COALESCE(CAST(`tp_condicao` AS STRING), ''), COALESCE(CAST(`desc_categoria_ledger_material` AS STRING), ''), COALESCE(CAST(`dt_sistema` AS STRING), ''), COALESCE(CAST(`dh_carga` AS STRING), ''), COALESCE(CAST(`dh_atualizacao` AS STRING), ''), COALESCE(CAST(`hash_conteudo` AS STRING), ''), COALESCE(CAST(`vl_estoque_real` AS STRING), ''), COALESCE(CAST(`vl_preco_unitario` AS STRING), ''), COALESCE(CAST(`dt_ingestao` AS STRING), '')))) AS linhas_100pct_identicas,
       CASE WHEN COUNT(*) = COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_area_avaliacao` AS STRING), ''), COALESCE(CAST(`cod_referencia_documento` AS STRING), ''), COALESCE(CAST(`tp_moeda_externa_perspectiva_avaliacao` AS STRING), ''), COALESCE(CAST(`cod_ledger_contabilidade_geral` AS STRING), ''), COALESCE(CAST(`cod_referencia_execucao_calculo_custos` AS STRING), ''), COALESCE(CAST(`cod_principio_contabil` AS STRING), ''), COALESCE(CAST(`num_calculo_custos_sem_estrutura_qtd` AS STRING), ''), COALESCE(CAST(`cod_ano_periodo_aaaappp_lancamento` AS STRING), ''), COALESCE(CAST(`cod_ano_periodo_origem_material_processo` AS STRING), ''), COALESCE(CAST(`cod_ano_corrente_periodo_yyyyppp_momento` AS STRING), ''), COALESCE(CAST(`ind_calculo_custos` AS STRING), ''), COALESCE(CAST(`cod_operacao_empresarial` AS STRING), ''), COALESCE(CAST(`tp_ledger_material_operacao` AS STRING), ''), COALESCE(CAST(`tp_item_documento_ledger_material` AS STRING), ''), COALESCE(CAST(`tp_exibicao_dados_ledger_material` AS STRING), ''), COALESCE(CAST(`tp_processo_alternativa_suprimento_consumo` AS STRING), ''), COALESCE(CAST(`cod_grupo_tipos_movimento_ledger_material` AS STRING), ''), COALESCE(CAST(`cod_alternativa_processo_suprimento` AS STRING), ''), COALESCE(CAST(`cod_processo_producao` AS STRING), ''), COALESCE(CAST(`num_calculo_custos_quantidade_produzida` AS STRING), ''), COALESCE(CAST(`qt_transacao_unidade_medida_avaliacao` AS STRING), ''), COALESCE(CAST(`vl_estoque_operacional` AS STRING), ''), COALESCE(CAST(`vl_diferenca_preco` AS STRING), ''), COALESCE(CAST(`vl_desvio_taxa_cambio` AS STRING), ''), COALESCE(CAST(`vl_diferenca_transferencia` AS STRING), ''), COALESCE(CAST(`vl_estoque_alternativo_interno_periodico` AS STRING), ''), COALESCE(CAST(`qt_estoque_avaliado` AS STRING), ''), COALESCE(CAST(`vl_estoque_avaliado_total_antes_transacao` AS STRING), ''), COALESCE(CAST(`vl_alternativo_estoque_avaliado_total_antes` AS STRING), ''), COALESCE(CAST(`vl_preco_padrao_antes_transacao` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco_antes_transacao` AS STRING), ''), COALESCE(CAST(`vl_venda_calculo_real_artigo` AS STRING), ''), COALESCE(CAST(`qt_limitacao_preco` AS STRING), ''), COALESCE(CAST(`vl_lucro_intercompanhias` AS STRING), ''), COALESCE(CAST(`vl_estoque_venda` AS STRING), ''), COALESCE(CAST(`vl_coeficiente_equivalencia_producao_conjunta` AS STRING), ''), COALESCE(CAST(`vl_preco_medio_movel_antes_transacao` AS STRING), ''), COALESCE(CAST(`dh_registro_hora` AS STRING), ''), COALESCE(CAST(`qt_pedido_unidade_medida_pedido` AS STRING), ''), COALESCE(CAST(`vl_entrada_fatura_ordem_producao` AS STRING), ''), COALESCE(CAST(`vl_liquidacao_ordem_producao` AS STRING), ''), COALESCE(CAST(`vl_estoque_operacional_centro_lucro` AS STRING), ''), COALESCE(CAST(`qt_referencia_fatura_liq_ordem_producao` AS STRING), ''), COALESCE(CAST(`vl_diferenca_preco_centro_lucro` AS STRING), ''), COALESCE(CAST(`sg_quantidade_avaliacao` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`ind_preco_desvio_taxa_cambio_relevantes` AS STRING), ''), COALESCE(CAST(`ind_desvios_sao_relevantes_logica_limitacao` AS STRING), ''), COALESCE(CAST(`ind_preco_interno_relevante_modificacao_centro` AS STRING), ''), COALESCE(CAST(`dt_lancamento_documento` AS STRING), ''), COALESCE(CAST(`num_ciclo_execucao` AS STRING), ''), COALESCE(CAST(`num_ordem` AS STRING), ''), COALESCE(CAST(`num_item_ordem` AS STRING), ''), COALESCE(CAST(`num_ledger_material` AS STRING), ''), COALESCE(CAST(`num_ledger_material_modificacao_preco_ano` AS STRING), ''), COALESCE(CAST(`num_item_ledger_material` AS STRING), ''), COALESCE(CAST(`cod_operacao_referencia` AS STRING), ''), COALESCE(CAST(`cod_sistema_logico_origem` AS STRING), ''), COALESCE(CAST(`sg_unidades_organizacionais_referencia` AS STRING), ''), COALESCE(CAST(`num_documento_referencia` AS STRING), ''), COALESCE(CAST(`num_partida_individual_referencia` AS STRING), ''), COALESCE(CAST(`num_grupo_partidas_individuais_referencia` AS STRING), ''), COALESCE(CAST(`nm_usuario` AS STRING), ''), COALESCE(CAST(`cod_transacao` AS STRING), ''), COALESCE(CAST(`cod_operacao_ledger_material` AS STRING), ''), COALESCE(CAST(`tp_transacao_pedido_historico_ordem_producao` AS STRING), ''), COALESCE(CAST(`cod_grupo_origem_classe_custo` AS STRING), ''), COALESCE(CAST(`cod_classe_custo_ordem` AS STRING), ''), COALESCE(CAST(`tp_movimento_administracao_estoques` AS STRING), ''), COALESCE(CAST(`cod_centro_ordem_producao_dados_historico` AS STRING), ''), COALESCE(CAST(`ind_estorno_apropriacao_ordem` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_ordem_producao` AS STRING), ''), COALESCE(CAST(`cod_periodo_apropriacao_custos_ordem` AS STRING), ''), COALESCE(CAST(`ind_capitalizar_diferencas_cambio` AS STRING), ''), COALESCE(CAST(`dt_envelhecimento_dados` AS STRING), ''), COALESCE(CAST(`cod_motivo_classificacao_contabil` AS STRING), ''), COALESCE(CAST(`cod_motivo_modificacao` AS STRING), ''), COALESCE(CAST(`cod_versao_migracao_dados` AS STRING), ''), COALESCE(CAST(`st_cobertura_estoque` AS STRING), ''), COALESCE(CAST(`cod_caso_fallback` AS STRING), ''), COALESCE(CAST(`tp_preco_fallback` AS STRING), ''), COALESCE(CAST(`cod_avaliacao_externa` AS STRING), ''), COALESCE(CAST(`ind_conexao_interrompida` AS STRING), ''), COALESCE(CAST(`cod_moeda_centro_lucro` AS STRING), ''), COALESCE(CAST(`num_documento_compras` AS STRING), ''), COALESCE(CAST(`num_item_documento_compras` AS STRING), ''), COALESCE(CAST(`num_exercicio_periodo_atual` AS STRING), ''), COALESCE(CAST(`num_documento_referencia_historico` AS STRING), ''), COALESCE(CAST(`num_item_documento_referencia_historico` AS STRING), ''), COALESCE(CAST(`tp_condicao` AS STRING), ''), COALESCE(CAST(`desc_categoria_ledger_material` AS STRING), ''), COALESCE(CAST(`dt_sistema` AS STRING), ''), COALESCE(CAST(`dh_carga` AS STRING), ''), COALESCE(CAST(`dh_atualizacao` AS STRING), ''), COALESCE(CAST(`hash_conteudo` AS STRING), ''), COALESCE(CAST(`vl_estoque_real` AS STRING), ''), COALESCE(CAST(`vl_preco_unitario` AS STRING), ''), COALESCE(CAST(`dt_ingestao` AS STRING), ''))))
            THEN 'OK - nenhuma linha totalmente identica'
            ELSE 'ATENCAO - existem linhas identicas em todos os campos' END AS veredito
FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128';

## 16. Amostra de linhas completas

In [0]:
-- 16. AMOSTRA
SELECT * FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
ORDER BY `num_ledger_material`, `num_item_ledger_material`
LIMIT 20;

## 17. Distribuição interna do centro 4128

Como o volume se reparte dentro do cenário. Útil para conferir se o extrato do SAP
tem a mesma composição.

In [0]:
-- 17. DISTRIBUICAO POR cod_ano_periodo_aaaappp_lancamento
SELECT COALESCE(NULLIF(trim(CAST(`cod_ano_periodo_aaaappp_lancamento` AS STRING)), ''), '(vazio)') AS `cod_ano_periodo_aaaappp_lancamento`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_ano_periodo_aaaappp_lancamento` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 17. DISTRIBUICAO POR tp_ledger_material_operacao
SELECT COALESCE(NULLIF(trim(CAST(`tp_ledger_material_operacao` AS STRING)), ''), '(vazio)') AS `tp_ledger_material_operacao`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`tp_ledger_material_operacao` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 17. DISTRIBUICAO POR cod_principio_contabil
SELECT COALESCE(NULLIF(trim(CAST(`cod_principio_contabil` AS STRING)), ''), '(vazio)') AS `cod_principio_contabil`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_principio_contabil` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 17. DISTRIBUICAO POR tp_movimento_administracao_estoques
SELECT COALESCE(NULLIF(trim(CAST(`tp_movimento_administracao_estoques` AS STRING)), ''), '(vazio)') AS `tp_movimento_administracao_estoques`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`tp_movimento_administracao_estoques` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

### 17.1 Freshness

In [0]:
-- 17.1 FRESHNESS
SELECT `dt_ingestao` AS data_carga, COUNT(*) AS linhas
FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
GROUP BY `dt_ingestao`
ORDER BY data_carga DESC
LIMIT 30;

## 18. Análises específicas — CKM3N

### 18.1 Validação das colunas derivadas

`vl_estoque_real` = `vl_estoque_operacional` + `vl_diferenca_preco` (STVAL + PRD).

In [0]:
-- 18.1 VALIDACAO DE vl_estoque_real
SELECT COUNT(*)                                                    AS linhas,
       COUNT_IF(ABS(vl_estoque_real
              - (COALESCE(vl_estoque_operacional,0)
               + COALESCE(vl_diferenca_preco,0))) <= 0.005)        AS formula_confere,
       COUNT_IF(ABS(vl_estoque_real
              - (COALESCE(vl_estoque_operacional,0)
               + COALESCE(vl_diferenca_preco,0))) > 0.005)         AS formula_diverge,
       MAX(ABS(vl_estoque_real
             - (COALESCE(vl_estoque_operacional,0)
              + COALESCE(vl_diferenca_preco,0))))                  AS maior_diferenca,
       CASE WHEN COUNT_IF(ABS(vl_estoque_real
                        - (COALESCE(vl_estoque_operacional,0)
                         + COALESCE(vl_diferenca_preco,0))) > 0.005) = 0
            THEN 'OK - formula confere' ELSE 'DIVERGE - investigar' END AS veredito
FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128';

### 18.2 Comportamento declarado no schema

O schema afirma que `dt_envelhecimento_dados` vem `00000000` em 100% das linhas
e que `cod_avaliacao_externa` só tem vazio ou `0`.

In [0]:
-- 18.2 CONFIRMACAO DO COMPORTAMENTO DECLARADO
SELECT 'dt_envelhecimento_dados' AS coluna,
       COUNT_IF(dt_envelhecimento_dados IS NULL) AS nulos,
       COUNT(*)                                   AS total,
       CASE WHEN COUNT_IF(dt_envelhecimento_dados IS NULL) = COUNT(*)
            THEN 'CONFIRMADO - 100% nula' ELSE 'DIVERGE - ha valores' END AS veredito
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
UNION ALL
SELECT 'cod_avaliacao_externa',
       COUNT_IF(trim(COALESCE(cod_avaliacao_externa,'')) IN ('', '0')),
       COUNT(*),
       CASE WHEN COUNT_IF(trim(COALESCE(cod_avaliacao_externa,'')) NOT IN ('', '0')) = 0
            THEN 'CONFIRMADO - apenas vazio ou 0' ELSE 'DIVERGE' END
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128';

### 18.3 Períodos de lançamento no cenário

In [0]:
-- 18.3 PERIODOS
SELECT cod_ano_periodo_aaaappp_lancamento,
       COUNT(*)                       AS linhas,
       COUNT(DISTINCT cod_material)   AS materiais,
       MIN(dt_lancamento_documento)   AS lancamento_min,
       MAX(dt_lancamento_documento)   AS lancamento_max
FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'
GROUP BY cod_ano_periodo_aaaappp_lancamento
ORDER BY cod_ano_periodo_aaaappp_lancamento DESC
LIMIT 40;

### 18.4 Hash de conteúdo

In [0]:
-- 18.4 HASH DE CONTEUDO
SELECT COUNT(*)                                  AS linhas,
       COUNT(DISTINCT hash_conteudo)             AS hashes_distintos,
       COUNT(*) - COUNT(DISTINCT hash_conteudo)  AS hashes_repetidos,
       CASE WHEN COUNT(*) = COUNT(DISTINCT hash_conteudo)
            THEN 'OK - conteudo unico'
            ELSE 'ATENCAO - registros com conteudo identico' END AS veredito
FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128';

## 19. EXTRAÇÃO COMPLETA — centro 4128

**Esta é a célula que você baixa para comparar com o SAP.**

Após executar, use **Download → CSV** no resultado.

> **Limites do Databricks:** a tela mostra até 10.000 linhas, mas o download em CSV
> vai além disso. Se o volume for muito grande, use a célula 19.1.

In [0]:
-- 19. EXTRACAO COMPLETA DO CENARIO
SELECT *
FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n
WHERE `cod_area_avaliacao` = '4128'
ORDER BY `num_ledger_material`, `num_item_ledger_material`;

### 19.1 Alternativa para volume grande _(opcional)_

In [0]:
-- 19.1 GRAVAR EXTRACAO EM TABELA (opcional)
-- CREATE OR REPLACE TABLE dev_procurement.corp_curated.extracao_ckm3n_4128 AS
-- SELECT * FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128';
--
-- SELECT COUNT(*) FROM dev_procurement.corp_curated.extracao_ckm3n_4128;
SELECT 'Descomente as linhas acima se precisar gravar a extracao em tabela' AS instrucao;

## 20. Resumo do cenário

**Copie esta saída** e envie ao agente junto com o notebook.

In [0]:
-- 20. RESUMO DO CENARIO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128'),
g AS (
  SELECT 'num_ledger_material' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_ledger_material` FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128')
UNION ALL
  SELECT 'num_ledger_material + num_item_ledger_material' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_ledger_material`, `num_item_ledger_material` FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128')
UNION ALL
  SELECT 'num_ledger_material + num_item_ledger_material + num_ledger_material_modificacao_preco_ano' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_ledger_material`, `num_item_ledger_material`, `num_ledger_material_modificacao_preco_ano` FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128')
UNION ALL
  SELECT 'cod_material + cod_area_avaliacao + cod_ano_periodo_aaaappp_lancamento' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_material`, `cod_area_avaliacao`, `cod_ano_periodo_aaaappp_lancamento` FROM dev_procurement.corp_curated.vw_ds_ctrl_ckm3n WHERE `cod_area_avaliacao` = '4128')
)
SELECT 'CENARIO' AS bloco, 'transacao' AS item, 'CKM3N' AS valor
UNION ALL SELECT 'CENARIO', 'tabela', 'vw_ds_ctrl_ckm3n'
UNION ALL SELECT 'CENARIO', 'tipo de objeto', 'VIEW'
UNION ALL SELECT 'CENARIO', 'coluna de recorte', 'cod_area_avaliacao'
UNION ALL SELECT 'CENARIO', 'filtro', 'cod_area_avaliacao = 4128'
UNION ALL SELECT 'CENARIO', 'linhas no cenario', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'CENARIO', 'colunas', '99'
UNION ALL
SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ',
              CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL
SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total),
                'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Baixar a **seção 19** em CSV — é a base do centro 4128 no Datalake.
2. Extrair a CKM3N no SAP com o filtro equivalente ao centro 4128.
3. Anotar a data e hora das duas extrações.
4. Enviar ao agente de validação: este notebook executado + os arquivos do SAP.

### Antes de comparar

- [ ] **Coluna de recorte confirmada** — `cod_area_avaliacao` é o campo correto? (seção 0)
- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP antes de classificar como erro (seção 6)
- [ ] Zeros à esquerda normalizados nos dois lados (seção 13)
- [ ] Tolerância de 0,005 aplicada em campos `double` (seção 10)
- [ ] Totais numéricos conferidos (seção 11)
- [ ] Booleanas: `true`/`false` no Datalake vs `X`/vazio no SAP (seção 9)
